# Okta AI Agent Onboarding: Machine Access and OPA

Register and configure an AI agent in Okta using a guided sequence of API
requests. The workflow establishes the agent's identity, group ownership,
AWS KMS public-key binding, machine access, and an Okta Privileged Access
(OPA) secret connection.

Work alongside your **Okta Admin Console**, running each cell in order and
reviewing the results at the indicated checkpoints. The notebook represents
the agent during the optional authentication and token-exchange validation.

## **How Colab represents your agent**

**Colab runs the code that stands in for your agent's runtime.** During
onboarding it uses the registration application's identity to create and
configure a real AI agent in your Okta tenant. During runtime validation it
uses that newly registered agent's identity and its KMS-backed signing key.

### **Onboarding, Steps 1–11**

**Colab:** Reads the KMS public key and configures the agent as the registration service; optionally uses the signed-in administrator for OPA.

**Services:** AWS supplies the public key. Okta saves and verifies the agent, ownership, key binding and access configuration.

### **Runtime A: machine caller**

**Colab:** Requests an access token targeting the agent's unique URL.

**Services:** The custom authorization server issues the caller's token.

### **Runtime B: registered agent**

**Colab:** Asks KMS to sign its assertion, then presents it with the caller token to Okta.

**Services:** KMS signs with the private key; Okta checks the delegation and resource connection and returns the OPA secret.

**These are real API calls, tokens and authorization decisions in your own
environment.** The demonstrated path is **machine caller → registered agent
→ connected OPA secret**. It covers onboarding and access validation; it does
not deploy an LLM, run two independent agents conversing, or use the secret
against a downstream application.
[Understand the machine-access and secret-exchange flow](https://developer.okta.com/docs/guides/ai-agent-token-exchange/secret/main/).

## **Prerequisites**

Before you begin, prepare the following resources in your environment:

1. An Okta tenant with the relevant AI Agents and OPA features enabled. These
   APIs include Beta capabilities, so confirm tenant availability. Use its
   canonical API URL, such as `https://your-org.oktapreview.com`, rather than
   its separate `https://your-org-admin.oktapreview.com` Admin Console URL.
2. An **API Services registration application** authorized to call
   management APIs. Its authentication method is public/private key
   (`private_key_jwt`), and its registered public key matches the private JWK
   you store in Colab Secrets. This notebook uses **bearer access tokens**;
   it does not send Demonstrating Proof-of-Possession (DPoP) proof headers.
   Use an approved service configuration compatible with this flow.
   Grant the scopes below and the required administrative role before the
   workflow. Current Beta registration/A2A documentation lists SUPER_ADMIN;
   a scope grant alone does not supply the necessary administrative role.
   The notebook does not alter global app-admin or role settings.
   If you do not have this application yet, follow **Prepare the registration
   application** below before completing Configuration.

   ```text
   okta.aiAgents.read
   okta.aiAgents.manage
   okta.apps.read
   okta.groups.read
   okta.authorizationServers.read
   okta.resourceServers.read
   okta.governance.resourceOwner.read
   okta.governance.resourceOwner.manage
   ```

   `okta.resourceServers.read` is required to check the asynchronous
   authorization-server association operation.
3. An existing **OIDC machine service application**, active with
   `client_credentials`. Find its application resource ID under Applications;
   Step 3 resolves its actual OAuth client ID. Reuse this same caller for
   additional agents. It is distinct from the registration service's role.
4. An existing **custom authorization server**, its issuer, and an existing
   active client-credentials access policy/rule for that caller. Any scopes
   requested during the optional proof must already exist. Reuse this same
   server across agents. This notebook does not create or modify AS policies,
   scopes or its global audience. If these resources are not yet configured,
   follow **First-time environment setup: machine access** below.
5. An existing **owner group with at least two members**. Copy its group ID
   from Directory → Groups, or supply an unambiguous exact group name.
6. An existing, enabled **asymmetric RSA SIGN_VERIFY AWS KMS key**, plus AWS
   access to `kms:DescribeKey` and `kms:GetPublicKey`. The optional proof also
   needs `kms:Sign` with `RSASSA_PKCS1_V1_5_SHA_256`. Supply the region and key
   ID/ARN from AWS KMS. The notebook neither creates nor exports a private key.
7. One **existing OPA secret** eligible to be connected to the agent. Its
   resource ORN can be selected from safe discovery metadata; never enter the
   secret value. If service OAuth cannot see/connect it, use your admin user
   through an existing public native OIDC app with Device Authorization
   enabled, user assignment, and grants for `okta.aiAgents.read` and
   `okta.aiAgents.manage`. For these Beta API operations, use a Super
   Administrator with the separate OPA security admin role. No direct
   vault-management credentials or new
   secret creation are required by this workflow.

## **Prepare the registration application**

This is a one-time setup in your own Okta tenant. The application gives the
notebook permission to onboard agents through Okta's management APIs. It is
separate from both the agent being registered and the machine-access caller.
If your administrator has already prepared it, use its client ID and matching
private JWK and continue to Configuration.

1. **Create the application.** Sign in to your Admin Console as a Super
   Administrator. Open **Applications → Applications → Create App
   Integration** (some consoles label the menu **Applications and Resources**).
   Select **API Services**, then name and save the app. If prompted, select
   **Classic experience** to reach this option.
   [Create an API Services app](https://developer.okta.com/docs/guides/implement-oauth-for-okta-serviceapp/main/#create-a-service-app-integration).
2. **Configure its signing key.** Under **General → Client Credentials → Edit**,
   select **Public key / Private key** and **Save keys in Okta**. Add your
   public RSA signing JWK, including its `kid`. For a lab, **Generate new key**
   can create the pair; copy the private key in **JWK** format before closing
   the dialog, then save. For production, use your organization's approved key
   generation process.
   [Signing-key setup](https://developer.okta.com/docs/guides/implement-oauth-for-okta-serviceapp/main/#generate-the-jwk-using-the-admin-console).
3. **Authorize management access.** In **Okta API Scopes**, grant the eight
   scopes listed above. In **Admin Roles → Edit assignments**, assign the
   required role described in the prerequisites and save.
   [Scope grants](https://developer.okta.com/docs/guides/implement-oauth-for-okta-serviceapp/main/#grant-allowed-scopes)
   · [Role assignments](https://developer.okta.com/docs/guides/implement-oauth-for-okta-serviceapp/main/#assign-admin-roles-to-the-oauth-20-service-app).
4. **Connect it to Colab.** Copy its **Client ID** into
   `REG_SERVICE_CLIENT_ID`. Store the matching private key as a single JSON
   JWK object in the Colab Secret `REG_SERVICE_PRIVATE_JWK`, and enable
   **Notebook access**. This registration key is separate from the agent's
   private key, which remains in AWS KMS.

**Checkpoint:** you have the client ID, matching private JWK, scope grants
and app role assignment. Step 2 requests the service's OAuth token; the
subsequent prerequisite checks verify access to the existing resources.

## **Identities and audiences**

- **Registration service:** Configures the agent using its own client ID and private JWK.
- **Existing machine caller:** Receives permission to obtain an access token targeting this agent.
- **New agent:** Its public JWK is in Okta; its corresponding private key remains in KMS.
- **Optional OPA user sign-in:** Authorizes existing-secret discovery/connection with your admin permissions.
- **Agent resource URL:** A unique HTTPS URL per agent; immutable after creation and used as its machine-access audience.
- **OAuth assertion audience:** The relevant token endpoint, not the agent resource URL or AS default audience.

An example resource URL shape is
`https://agents.your-company.example/research-assistant-01`. Use your intended
unique value. No running HTTP service at that URL is needed for this
onboarding workflow. Choose the URL before creating the agent: it must be
unique and cannot be changed after registration.

## **Complete registration flow**

1. **Read the agent's public key from its existing KMS signing key.**
2. **Get the registration service's OAuth access token.**
3. **Verify the existing machine application.** Complete the read-only AS,
   owner, collision and OPA prerequisites in **Steps 3A–3D**.
4. **Create the agent with name, description and resource URL.**
5. **Resolve creation and retrieve the real agent ID.**
6. **Add and verify the KMS public JWK.**
7. **Discover the organization ID.**
8. **Resolve and verify the selected owner group.**
9. **Assign and verify ownership.**

**Complete access configuration before activation:**

- **9A:** Associate the existing custom AS while the agent is STAGED.
- **9B:** Resolve and verify its asynchronous association.
- **9C:** Authorize the existing machine caller through delegation.
- **9D:** Connect and verify one existing OPA secret.

**Step 10** activates the configured agent and confirms its status.
**Step 11** verifies the complete onboarding configuration. Keep the agent
staged until its access configuration and resource connection are ready.

An **optional runtime validation** follows: the machine caller obtains its
token, and Colab represents the agent by signing through KMS and performing
the OPA exchange. An optional appendix demonstrates an additional outgoing
custom authorization-server connection.

### **Running the notebook**

Each numbered step displays the API endpoint, request payload and expected
result. Shared helpers handle HTTP requests, signing and response validation.
Run the cells in order and review each result before continuing.

Store credentials in Colab Secrets rather than form fields or code cells.
The notebook displays configuration metadata and validation results while
keeping tokens, private keys and secret values out of its outputs.

---


## **Prepare your other resources**

**Already have the prerequisites? Continue to Configuration.** Otherwise,
complete the applicable setup below with your Okta, OPA or AWS administrator.
These are preparation tasks outside the notebook's agent-onboarding steps.
The workflow reuses these resources when you register additional agents.

### **First-time environment setup: machine access**

Complete this setup once in your own **Okta Admin Console** before running
the agent-registration cells. The caller application and custom
authorization server can then be reused when onboarding additional agents.
If both already exist, review the scope and policy settings below and
continue to **Setup checkpoint**. Example names and URLs are placeholders;
use values appropriate for your environment.

#### **1. Confirm that your environment supports this workflow**

Confirm the AI Agents and OPA availability listed in **Prerequisites**.
Machine access also requires a **custom authorization server**. In
production, custom authorization servers require **API Access Management**;
an Okta Integrator Free Plan org includes a custom server named `default`
for testing. Its initial configuration may not include an access policy.
Open **Security → API → Authorization Servers**. If this capability is
unavailable, have your Okta administrator confirm the subscription and
tenant configuration before continuing.
[Authorization server types and availability](https://developer.okta.com/docs/concepts/auth-servers/).

**Checkpoint:** your administrator can select or create a custom
authorization server. The **org authorization server** used by Step 2 to
obtain management-API tokens is separate from the custom server used by
the machine caller.

#### **2. Create the machine caller application**

Open **Applications → Applications → Create App Integration** (some
consoles use **Applications and Resources**). Select **API Services**, then
name and save the application; for example, **AI Agent Machine Caller**.
If prompted, choose **Classic experience** to reach this option. Verify
that the application is active and **Client Credentials** is enabled.
[Create the caller application](https://developer.okta.com/docs/guides/implement-grant-type/clientcreds/main/#set-up-your-app).

Record its **application resource ID** from the Admin Console URL for
`MACHINE_APP_ID`. Record its **OAuth Client ID** and configured
client-authentication method from **General → Client Credentials**;
Step 3 retrieves and verifies these details. For a caller using a client
secret, store that secret in Colab Secrets as `MACHINE_CLIENT_SECRET` when
running runtime validation. For `private_key_jwt`, use the matching private
JWK as `MACHINE_PRIVATE_JWK` instead. Follow **Set up Colab Secrets** below.

**Checkpoint:** the active caller supports Client Credentials, and you
have its application ID and the appropriate runtime credential. This is
a separate application from the registration service prepared above;
it does not need that service's management scopes or administrative role.

#### **3. Select or create the shared custom authorization server**

Under **Security → API → Authorization Servers**, select an approved
existing server, including `default` if appropriate. If you need a new
server, click **Add Authorization Server** and enter:

- **Name:** for example, `AI Agent Machine Access`.
- **Audience:** your intended shared API audience; for example,
  `https://api.your-company.example/agent-access`.
- **Description:** for example, `Issues machine-caller access tokens for AI agents`.

Save the server and confirm that it is active. From its **Settings**,
record the **Issuer**. An issuer of
`https://your-org.okta.com/oauth2/aus...` gives an `AUTHZ_SERVER_ID` of
`aus...`; an issuer ending in `/oauth2/default` gives an ID of `default`.
[Create a custom authorization server](https://developer.okta.com/docs/guides/customize-authz-server/main/#create-an-authorization-server).

**Checkpoint:** you have an active custom server and its ID. The shared
server's **Audience** is its default token audience. Each agent's
`AGENT_RESOURCE_URL` is configured separately during registration and is
used when requesting a token for that agent. Keep the shared server's
global audience unchanged when registering additional agents.

#### **4. Define the machine-access scope**

On the selected server, open **Scopes → Add Scope**. For this demonstration,
create a scope named `agent.invoke`, with an appropriate display phrase and
description, such as `Invoke a registered AI agent`. Use a scope suitable
for unattended Client Credentials requests; this flow has no end user to
provide consent. If an approved scope already exists, you can use it
instead. Enter the exact scope name in `RUNTIME_SCOPES`.
[Create custom scopes](https://developer.okta.com/docs/guides/customize-authz-server/main/#create-scopes).

**Checkpoint:** the requested scope exists on the selected server.
`agent.invoke` is an example custom scope, not a built-in Okta management
scope or an automatic permission grant.

#### **5. Configure the caller's access policy and rule**

Open **Access Policies → Add Policy** on that server. For example, name it
**AI Agent Machine Caller Policy** and select **The following clients**,
choosing the caller application created above. Save the policy. Add a rule
with the following conditions:

- **Grant type:** `Client Credentials`.
- **User:** `No user`.
- **Scopes requested:** the selected custom scope, such as `agent.invoke`.
- **Access token lifetime:** a value approved for your environment;
  for a lab, one hour is an example.

Save and confirm that both the policy and rule are active. Review their
priority against other policies and rules: Okta applies the first matching
policy and rule. If reusing an existing server, have its administrator
confirm that the intended rule actually applies to this caller.
[Create access policies](https://developer.okta.com/docs/guides/customize-authz-server/main/#create-access-policies)
· [Create policy rules](https://developer.okta.com/docs/guides/customize-authz-server/main/#create-rules-for-each-access-policy).

**Checkpoint:** the selected caller can request the selected scope using
Client Credentials without a user. A scope definition alone does not
authorize token issuance.

#### **6. Complete Configuration and choose the agent audience**

Copy the caller's application ID into `MACHINE_APP_ID`, the custom-server
ID into `AUTHZ_SERVER_ID`, and the allowed scope into `RUNTIME_SCOPES`.
Choose a distinct HTTPS `AGENT_RESOURCE_URL` for the new agent; for example,
`https://agents.your-company.example/research-assistant-01`. Choose another
URL for each additional agent. This identifier cannot be changed after
registration. No running HTTP service at that URL is required for this
demonstration. The registration cells create the agent and associate it
with your selected server and caller.
[Machine-access configuration](https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-add-manually.htm).

**Setup checkpoint:** you have the caller app ID, custom-server ID, allowed
scope, a unique agent resource URL and the runtime credential in Colab
Secrets. Complete the owner group, KMS and OPA preparation below, then
continue to **Configuration**. Step 3A checks the server, scope and policy
prerequisites. After onboarding passes, **Runtime A** verifies actual
caller-token issuance, signature and audience; **Runtime B** verifies the
agent's KMS-signed exchange for its connected OPA secret. Configuration
checks alone do not establish successful runtime access.

### **Owner group and AWS KMS**

Select an owner group under **Directory → Groups** with at least two members.
Copy its ID or exact name into Configuration. If needed, first
[create a group](https://help.okta.com/oie/en-us/content/topics/users-groups-profiles/usgp-groups-create.htm)
and add its intended owners. See
[agent ownership and client registration](https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-add-manually.htm).

For a new KMS key, follow
[Create an asymmetric KMS key](https://docs.aws.amazon.com/kms/latest/developerguide/asymm-create-key.html)
and choose **Asymmetric → Sign and verify → RSA**, such as `RSA_2048`.
Record its region and key ARN. The AWS identity used in Colab needs the
specific key permissions listed in Prerequisites, including `kms:Sign` for
runtime proof. Obtain an approved, limited AWS session and put its three
credential values in Colab Secrets.
[Temporary AWS credentials](https://docs.aws.amazon.com/IAM/latest/UserGuide/id_credentials_temp_use-resources.html).
If an approved AWS CLI profile already provides that limited temporary
session, run this in your local terminal, replacing `your-kms-profile`:

```bash
aws configure export-credentials --profile your-kms-profile --format env-no-export
```

Copy the three AWS values into the matching Colab Secrets. This command
displays credentials, so run it locally rather than in a shared notebook.
[AWS CLI credential export](https://docs.aws.amazon.com/cli/latest/reference/configure/export-credentials.html)
· [KMS key-policy permissions](https://docs.aws.amazon.com/kms/latest/developerguide/key-policies.html).
Signing-key creation and credential issuance happen before the workflow;
the notebook reads the existing key and never exports its private material.

### **Existing OPA secret and optional administrator sign-in**

Have your OPA administrator prepare one secret before starting. Select it
through Step 3B's metadata discovery; enter its ORN, never its value.
If none exists, first prepare a secret folder and save a lab secret in OPA.
[Secret folders](https://help.okta.com/oie/en-us/content/topics/privileged-access/pam-secrets-folder.htm)
· [Create a secret](https://help.okta.com/oie/en-us/content/topics/privileged-access/pam-enduser.htm).
[Resource prerequisites](https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-secure.htm)
· [Connect an agent to a secret](https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-connected-resource.htm).

If the registration service cannot discover or connect that secret, prepare
a separate **OIDC Native Application** for administrator sign-in:

1. Create or select the native app, enable **Device Authorization**, and use
   client authentication **None** (a public client).
2. Assign the intended administrator under **Assignments**. In **Okta API
   Scopes**, grant `okta.aiAgents.read` and `okta.aiAgents.manage`.
3. Copy its Client ID into `OPA_DEVICE_CLIENT_ID`, select
   `OPA_AUTH_MODE=user_device`, and follow Steps 3C–3D. Use a Super
   Administrator who also has OPA security admin access.

[Native OIDC app setup](https://help.okta.com/en-us/Content/Topics/Apps/Apps_App_Integration_Wizard_OIDC.htm)
· [Assign the intended user](https://help.okta.com/oie/en-us/content/topics/provisioning/lcm/lcm-assign-app-user.htm)
· [Grant Okta API scopes](https://developer.okta.com/docs/guides/implement-oauth-for-okta/main/#define-allowed-scopes)
· [Device Authorization setup](https://developer.okta.com/docs/guides/device-authorization-grant/main/)
· [OPA roles](https://help.okta.com/oie/en-us/content/topics/privileged-access/pam-roles.htm).
This notebook uses the **org authorization server** for this sign-in and
requests the two Okta management scopes above. The device guide's custom-AS
policy and `offline_access` examples are not additional requirements here.
This sign-in authorizes OPA discovery/connection; the agent later uses its
own KMS-backed identity for runtime exchange.
Ask your PAM administrator to confirm Security admin access through the
appropriate OPA group.
[OPA user and group role administration](https://help.okta.com/oie/en-us/content/topics/privileged-access/pam-users-and-groups.htm).

**Before creating the agent:** run the Configuration, Dependencies and
Helpers cells, then Steps 1–3B. Complete 3C–3D when prompted, and confirm
the selected OPA secret passes discovery. Let the notebook create the new
agent in Step 4; use the Admin Console to inspect the same agent afterward.

---


## **Configuration — your existing resources**

Fill in the native Colab form below before running Step 1. Only public
identifiers and descriptive values belong here. **MACHINE_APP_ID** is
the application's Okta resource ID; Step 3 reads its OAuth client ID.
**REG_SERVICE_CLIENT_ID** is the separate API Services client's OAuth ID.

Supply either the owner group's ID or its exact name. If you do not
know the existing OPA secret ORN, leave it blank initially: Step 3B
lists safe metadata. Copy the intended ORN into this form, rerun this
cell and Steps 1–3B before creating the agent. Configuration reruns
retain saved agent/operation IDs. After creation begins, resource
settings are locked to prevent changing the identity under a live run.

### **Set up Colab Secrets**

Complete this setup in your own Google account before running the
notebook. Use credentials for your environment, obtained through
your administrator's approved process.

**`REG_SERVICE_PRIVATE_JWK` — Onboarding**

The **private** signing JWK saved when preparing the registration API Services app above. Paste one JSON key object, including `kid` and private fields, rather than a public key, PEM file or `keys` array.

**`AWS_ACCESS_KEY_ID` — Onboarding**

Access key ID from an approved temporary AWS session authorized for the selected KMS key.

**`AWS_SECRET_ACCESS_KEY` — Onboarding**

Secret access key from that same temporary AWS session.

**`AWS_SESSION_TOKEN` — Onboarding**

Session token from that same temporary AWS session. Copy all three AWS values together and check their expiry.

**`MACHINE_CLIENT_SECRET` — Optional runtime validation**

The existing machine caller's client secret, available to its administrator under **General → Client Credentials**. Use this for a caller configured with `client_secret_basic` or `client_secret_post`.

**`MACHINE_PRIVATE_JWK` — Optional runtime validation**

The existing machine caller's matching private signing JWK, saved when its public key was registered. Use this instead of `MACHINE_CLIENT_SECRET` for a caller configured with `private_key_jwt`.

[Registration signing-key setup](https://developer.okta.com/docs/guides/implement-oauth-for-okta-serviceapp/main/#generate-the-jwk-using-the-admin-console)
· [Machine caller setup](https://developer.okta.com/docs/guides/implement-grant-type/clientcreds/main/#set-up-your-app)
· [Export an approved AWS session](https://docs.aws.amazon.com/cli/latest/reference/configure/export-credentials.html).

1. **Open Secrets:** select the **key icon** in Colab's left sidebar.
2. **Add a secret:** click **Add new secret**. Enter its exact name
   from the list in **Name**, then paste its credential in **Value**.
   Copy the value itself, without a shell variable name, surrounding
   quotation marks or an `export` prefix.
3. **Allow this notebook to read it:** enable **Notebook access**
   for that entry. Repeat for all four onboarding entries. Add the
   caller credential only when you intend to run runtime validation.
4. **Check setup:** confirm the names and access switches, then close
   the Secrets pane. Complete the public form below and run
   **Configuration → Dependencies → Helpers → Step 1 → Step 2**.
   Step 1 validates KMS access; Step 2 validates registration OAuth.

**You do not need to obtain or paste an Okta access token.** Step 2
requests it using the registration application's private JWK.
The agent's own private key remains in KMS; Colab reads its public
key and asks KMS to sign during optional runtime validation.

**If setup fails:** check the exact secret name and its Notebook
access switch. For expired AWS credentials, replace all three values
with a new approved session for the same key. If the registration
private JWK was not retained, contact the app administrator to
prepare a replacement signing key before continuing.
Never paste credentials into code, the public form or cell output.

**Where to find the public configuration values**

- **`OKTA_DOMAIN`:** Your canonical org URL; omit the Admin Console's `-admin` suffix.
- **`REG_SERVICE_CLIENT_ID`:** General tab of the registration API Services app.
- **`MACHINE_APP_ID`:** The existing caller's application resource ID, visible in its Admin Console URL. Step 3 reads its OAuth client ID.
- **`AUTHZ_SERVER_ID`:** The selected custom server's issuer, after `/oauth2/`; `default` is also valid.
- **`OWNER_GROUP_ID` or `OWNER_GROUP_NAME`:** Directory → Groups: use the group's ID or exact unique name.
- **`AWS_REGION` / `AWS_KMS_KEY_ID`:** Region and key ARN of the existing RSA signing key in AWS KMS.
- **`OPA_SECRET_ORN`:** The exact selected secret ORN returned by Step 3B metadata discovery.
- **`OPA_DEVICE_CLIENT_ID`:** Client ID of the optional public native sign-in app; needed only for `user_device`.
- **`RUNTIME_SCOPES`:** Existing custom-AS scope names allowed for the machine caller, separated by spaces.

Run the numbered cells deliberately, following their outputs. Runtime,
the additional managed-connection appendix and deletion default to off.

---


In [ ]:
# @title Configuration { display-mode: "form" }
# @markdown Your tenant and existing registration service
OKTA_DOMAIN = "" # @param {type:"string"}
REG_SERVICE_CLIENT_ID = "" # @param {type:"string"}
# @markdown New agent profile; resource URL is unique and immutable
AGENT_NAME = "" # @param {type:"string"}
AGENT_DESCRIPTION = "" # @param {type:"string"}
AGENT_RESOURCE_URL = "" # @param {type:"string"}
# @markdown Existing owner group (ID preferred, or exact name)
OWNER_GROUP_ID = "" # @param {type:"string"}
OWNER_GROUP_NAME = "" # @param {type:"string"}
# @markdown Reuse this existing machine caller and custom authorization server
MACHINE_APP_ID = "" # @param {type:"string"}
AUTHZ_SERVER_ID = "" # @param {type:"string"}
# @markdown Existing asymmetric RSA signing key in AWS KMS
AWS_REGION = "" # @param {type:"string"}
AWS_KMS_KEY_ID = "" # @param {type:"string"}
# @markdown Existing OPA secret, and optional administrator sign-in
OPA_SECRET_ORN = "" # @param {type:"string"}
OPA_AUTH_MODE = "registration_service" # @param ["registration_service", "user_device"]
OPA_DEVICE_CLIENT_ID = "" # @param {type:"string"}
# @markdown Optional runtime proof uses existing custom-AS scopes
RUNTIME_SCOPES = "" # @param {type:"string"}
RUN_RUNTIME_PROOF = False # @param {type:"boolean"}
# @markdown Optional appendix has its own existing custom-AS scope
OPTIONAL_CUSTOM_AS_SCOPE = "" # @param {type:"string"}
RUN_OPTIONAL_CUSTOM_AS_CONNECTION = False # @param {type:"boolean"}
# @markdown Delete ONLY this notebook's newly created agent after inspection
DELETE_CREATED_AGENT = False # @param {type:"boolean"}

OKTA_DOMAIN = OKTA_DOMAIN.strip().rstrip("/")
if "guard_config" in globals():
    guard_config()
print("Public configuration loaded. No credentials or resource IDs were reset.")


## **Dependencies**

Run this once in a Python 3.11+ Colab runtime (including Python 3.12).
These are public packages for direct HTTP, JWT verification and AWS.
The notebook is self-contained; no other project files are uploaded.

---


In [ ]:
import sys
if sys.version_info < (3, 11):
    raise RuntimeError("Use a Python 3.11 or newer Colab runtime.")
%pip -q install requests==2.32.4 PyJWT==2.15.1 cryptography==50.0.2 'boto3[crt]==1.43.108'


## **Small, visible helpers**

These helpers handle HTTP requests, signing and response validation.
Each step below still shows its actual endpoint and request payload.
HTTP output contains method/path/status only; tokens, authorization
headers, private keys, raw error bodies and secret values are hidden.

A resource ID identifies a saved object. An **operation ID** identifies
asynchronous work and is not the resulting agent ID. The **org ID**
identifies the tenant inside Okta resource names (ORNs). An ORN also
names a service and resource type; it is not an OAuth client ID.

---


In [ ]:
import base64
import hashlib
import json
import re
import time
import uuid
from urllib.parse import urljoin, urlsplit
import boto3
import jwt
import requests
from botocore.exceptions import BotoCoreError, ClientError
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric import rsa

class LabError(RuntimeError):
    """A safe lesson error with no response body or credential attached."""

CONFIG_FIELDS = (
    "OKTA_DOMAIN", "REG_SERVICE_CLIENT_ID", "AGENT_NAME",
    "AGENT_DESCRIPTION", "AGENT_RESOURCE_URL", "OWNER_GROUP_ID",
    "OWNER_GROUP_NAME", "MACHINE_APP_ID", "AUTHZ_SERVER_ID",
    "AWS_REGION", "AWS_KMS_KEY_ID", "OPA_SECRET_ORN",
)
for _name, _initial in {
    "PREFLIGHT_CHECKS": {}, "LOCKED_CONFIG_FINGERPRINT": None,
    "CREATE_ATTEMPTED": False, "AGENT_ID": None,
    "AGENT_CLIENT_ID": None, "CREATION_OPERATION_LOCATION": None,
    "CREATED_BY_THIS_NOTEBOOK": False, "HEADERS": {},
    "OPA_HEADERS": {}, "OPA_AUTH_ORIGIN": None,
    "REG_AUTH_ORIGIN": None, "DESIGN_TIME_PASSED": False,
    "RUNTIME_RESULT": {"status": "NOT_RUN"},
}.items():
    globals().setdefault(_name, _initial)

def config_fingerprint():
    values = {name: globals()[name] for name in CONFIG_FIELDS}
    return hashlib.sha256(json.dumps(values, sort_keys=True).encode()).hexdigest()

def guard_config():
    if LOCKED_CONFIG_FINGERPRINT and config_fingerprint() != LOCKED_CONFIG_FINGERPRINT:
        raise LabError("Resource settings changed after creation began. Restore the reviewed configuration; do not reuse this run for a different agent.")
    parsed = urlsplit(OKTA_DOMAIN)
    if not re.fullmatch(r"https://[a-z0-9-]+\.(okta|oktapreview|okta-emea)\.com", OKTA_DOMAIN) or (parsed.hostname or "").split(".")[0].endswith("-admin"):
        raise LabError("Use the canonical HTTPS Okta org URL, not its Admin Console URL or a custom hostname.")

def mark_preflight(name):
    PREFLIGHT_CHECKS[name] = config_fingerprint()

def invalidate_preflight(name):
    global DESIGN_TIME_PASSED, RUNTIME_RESULT
    PREFLIGHT_CHECKS.pop(name, None)
    DESIGN_TIME_PASSED = False
    RUNTIME_RESULT = {"status": "NOT_RUN", "reason": "prerequisites_rechecked"}

def require_preflight(*names):
    guard_config()
    if any(PREFLIGHT_CHECKS.get(name) != config_fingerprint() for name in names):
        raise LabError("Prerequisites are missing or configuration changed. Rerun Steps 1–3B before creation.")

def trusted_url(path):
    guard_config()
    url = urljoin(OKTA_DOMAIN + "/", path)
    parsed = urlsplit(url)
    if parsed.scheme != "https" or parsed.netloc != urlsplit(OKTA_DOMAIN).netloc or parsed.username or parsed.password or parsed.fragment:
        raise LabError("Refusing a request or redirect outside the configured Okta org.")
    return url

def get_secret(name, required=True):
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = ""
    if not isinstance(value, str) or not value.strip():
        if required:
            raise LabError("Missing Colab Secret " + name + ". Add it using the key icon and enable Notebook access.")
        return ""
    return value

def okta_request(method, path, headers=None, json=None, data=None, allowed=(200,), params=None):
    url = trusted_url(path)
    request_headers = dict(HEADERS if headers is None else headers)
    if request_headers.get("Authorization") and REG_AUTH_ORIGIN != OKTA_DOMAIN:
        raise LabError("The org changed; run service authentication again before using any cached credentials.")
    try:
        response = requests.request(method, url, headers=request_headers, json=json,
            data=data, params=params, timeout=30, allow_redirects=False)
    except requests.RequestException:
        raise LabError("HTTP transport failed. For a write, inspect its saved state before retrying; no automatic POST retry occurred.") from None
    print(f"[HTTP] {method.upper()} {urlsplit(url).path} → {response.status_code}")
    if response.status_code not in allowed:
        raise LabError(f"Request failed (HTTP {response.status_code}). Check this step's required permissions and inputs. Response body and credentials are hidden.")
    return response

def response_json(response):
    try:
        value = response.json()
    except (ValueError, UnicodeError):
        raise LabError("Expected a JSON response; raw body is hidden.") from None
    if not isinstance(value, dict):
        raise LabError("Expected a JSON object from this endpoint.")
    return value

def list_all(path, headers=None, params=None):
    rows, seen = [], set()
    for _page in range(100):
        url = trusted_url(path)
        fingerprint = (url, json.dumps(params, sort_keys=True))
        if fingerprint in seen:
            raise LabError("Pagination loop detected.")
        seen.add(fingerprint)
        response = okta_request("GET", url, headers=headers, params=params)
        try:
            body = response.json()
        except ValueError:
            raise LabError("List response was not valid JSON.") from None
        batch = body if isinstance(body, list) else body.get("data", body.get("keys")) if isinstance(body, dict) else None
        if not isinstance(batch, list) or any(not isinstance(row, dict) for row in batch):
            raise LabError("Expected a list, data array or keys array.")
        rows.extend(batch)
        following = response.links.get("next", {}).get("url")
        if not following and isinstance(body, dict):
            following = body.get("_links", {}).get("next", {}).get("href")
        if not following:
            return rows
        path, params = trusted_url(following), None
    raise LabError("Pagination exceeded the 100-page bound.")

def poll_operation(location, headers=None):
    url = trusted_url(location)
    if not urlsplit(url).path.startswith("/workload-principals/api/v1/operations/"):
        raise LabError("This helper handles workload operations only; the machine step polls its own resource operation.")
    deadline = time.monotonic() + 90
    while time.monotonic() < deadline:
        response = okta_request("GET", url, headers=headers)
        operation = response_json(response)
        status = operation.get("status")
        print("Operation status:", status if status in {"SCHEDULED", "IN_PROGRESS", "COMPLETED", "FAILED"} else "unrecognized")
        if status == "COMPLETED":
            return operation
        if status not in {"SCHEDULED", "IN_PROGRESS"}:
            raise LabError("Operation failed or returned an unsupported status. Inspect the known operation before retrying.")
        try:
            delay = max(2, int(response.headers.get("Retry-After", "2")))
        except ValueError:
            delay = 2
        time.sleep(min(delay, max(0, deadline - time.monotonic())))
    raise LabError("Operation timed out. Retain its Location and inspect before repeating a write.")

def orn(service, resource):
    if not globals().get("ORG_ID") or not globals().get("PARTITION"):
        raise LabError("Run Step 7 to resolve the organization identity first.")
    return f"orn:{PARTITION}:{service}:{ORG_ID}:{resource}"

def client_assertion(client_id, audience, private_jwk):
    audience_url = urlsplit(audience)
    if audience_url.scheme != "https" or not audience_url.hostname or audience_url.username or audience_url.password or audience_url.query or audience_url.fragment:
        raise LabError("A client assertion needs a valid HTTPS token-endpoint audience.")
    try:
        key = json.loads(private_jwk) if isinstance(private_jwk, str) else private_jwk
        if not isinstance(key, dict) or key.get("kty") != "RSA" or not all(key.get(field) for field in ("kid", "n", "e", "d")):
            raise ValueError("invalid key")
        private = jwt.algorithms.RSAAlgorithm.from_jwk(json.dumps(key))
        now = int(time.time())
        return jwt.encode({"iss": client_id, "sub": client_id, "aud": audience,
            "iat": now, "exp": now + 60, "jti": str(uuid.uuid4())}, private,
            algorithm="RS256", headers={"kid": key["kid"]})
    except Exception:
        raise LabError("Private JWK is invalid or cannot sign RS256; its contents are hidden.") from None

def kms_public_jwk(der, kid=None):
    public = serialization.load_der_public_key(der)
    if not isinstance(public, rsa.RSAPublicKey) or public.key_size < 2048:
        raise LabError("KMS must provide an RSA public key of at least 2048 bits.")
    numbers = public.public_numbers()
    def encode_integer(value):
        return base64.urlsafe_b64encode(value.to_bytes((value.bit_length() + 7) // 8, "big")).rstrip(b"=").decode()
    return {"kty": "RSA", "alg": "RS256", "use": "sig",
        "kid": kid or hashlib.sha256(der).hexdigest()[:32],
        "n": encode_integer(numbers.n), "e": encode_integer(numbers.e)}

def same_public_jwk(actual, expected):
    return all(actual.get(k) == expected.get(k) for k in ("kid", "kty", "alg", "n", "e"))

def new_kms_client():
    access = get_secret("AWS_ACCESS_KEY_ID", required=False)
    secret = get_secret("AWS_SECRET_ACCESS_KEY", required=False)
    session = get_secret("AWS_SESSION_TOKEN", required=False)
    if any((access, secret, session)) and not (access and secret):
        raise LabError("Supply both AWS access/secret keys and the session token for temporary credentials.")
    try:
        aws = boto3.Session(aws_access_key_id=access or None,
            aws_secret_access_key=secret or None, aws_session_token=session or None,
            region_name=AWS_REGION)
        return aws.client("kms")
    except (BotoCoreError, ClientError):
        raise LabError("AWS credentials could not initialize KMS; details hidden.") from None

guard_config()
print("Readable helpers ready. Existing agent and operation IDs were retained.")


## **Step 1: Get the agent's public key from AWS KMS**

Retrieve the public key from an **existing, enabled RSA signing key
in AWS KMS**. The agent's private key remains in KMS and is used to
sign its authentication assertion during runtime validation.

`DescribeKey` resolves the key's Amazon Resource Name (ARN), and
`GetPublicKey` retrieves its public material. The notebook retains
that ARN and public JWK to verify the same key throughout this run.

**Expected result:** the KMS key ARN, public-key identifier and a PASS
result. This read-only step does not modify Okta. The registration
service uses its own separate private JWK.

[KMS public-key retrieval](https://docs.aws.amazon.com/kms/latest/APIReference/API_GetPublicKey.html).

---


In [ ]:
guard_config()
invalidate_preflight("kms")
if not AWS_REGION or not AWS_KMS_KEY_ID:
    raise LabError("Enter the existing KMS key ID/ARN and AWS region in Configuration.")
KMS_CLIENT = new_kms_client()
try:
    key_metadata = KMS_CLIENT.describe_key(KeyId=AWS_KMS_KEY_ID)["KeyMetadata"]
    if key_metadata.get("KeyState") != "Enabled" or key_metadata.get("KeyUsage") != "SIGN_VERIFY" or not str(key_metadata.get("KeySpec", "")).startswith("RSA_"):
        raise LabError("KMS key must be enabled, RSA, and SIGN_VERIFY.")
    resolved_arn = key_metadata["Arn"]
    key_response = KMS_CLIENT.get_public_key(KeyId=resolved_arn)
except (BotoCoreError, ClientError):
    raise LabError("KMS DescribeKey/GetPublicKey failed. Check the existing key, region and AWS permissions.") from None
if key_response.get("KeyId") != resolved_arn or key_response.get("KeyUsage") != "SIGN_VERIFY" or "RSASSA_PKCS1_V1_5_SHA_256" not in key_response.get("SigningAlgorithms", []):
    raise LabError("KMS public key identity or RS256 signing capability did not match.")
selected_public_jwk = kms_public_jwk(key_response["PublicKey"])
if globals().get("KMS_KEY_ARN") and (CREATE_ATTEMPTED or globals().get("KMS_SELECTION_CONTEXT") == config_fingerprint()) and (KMS_KEY_ARN != resolved_arn or not same_public_jwk(PUBLIC_JWK, selected_public_jwk)):
    raise LabError("The selected KMS key changed. Deliberate key rotation is outside this notebook.")
KMS_KEY_ARN, PUBLIC_JWK = resolved_arn, selected_public_jwk
KMS_SELECTION_CONTEXT = config_fingerprint()
KEY_ID = PUBLIC_JWK["kid"]
mark_preflight("kms")
print({"kms_key_arn": KMS_KEY_ARN, "public_key_kid": KEY_ID, "kms_public_key": "PASS", "kms_signing": "NOT_RUN"})


## **Step 2: Get Access Token — service OAuth / private_key_jwt**

The existing registration service signs its authentication assertion,
whose audience is the **org token
endpoint**, and asks for management scopes. Scope grants and the app's
admin role must already allow these operations. This does not use the
agent's KMS key or the machine caller's credential.

Expected: token type/lifetime only. The bearer token stays in memory.
Re-run this step to renew the same service login without losing the
current agent ID. It preserves separately authorized same-org OPA login.

[Service-app OAuth setup](https://developer.okta.com/docs/guides/implement-oauth-for-okta-serviceapp/main/).

---


In [ ]:
guard_config()
if not REG_SERVICE_CLIENT_ID:
    raise LabError("Set the registration service OAuth client ID first.")
REGISTRATION_SCOPES = (
    "okta.aiAgents.read", "okta.aiAgents.manage", "okta.apps.read",
    "okta.groups.read", "okta.authorizationServers.read", "okta.resourceServers.read",
    "okta.governance.resourceOwner.read", "okta.governance.resourceOwner.manage",
)
token_endpoint = trusted_url("/oauth2/v1/token")
token_data = {
    "grant_type": "client_credentials",
    "client_id": REG_SERVICE_CLIENT_ID,
    "client_assertion_type": "urn:ietf:params:oauth:client-assertion-type:jwt-bearer",
    "client_assertion": client_assertion(REG_SERVICE_CLIENT_ID, token_endpoint, get_secret("REG_SERVICE_PRIVATE_JWK")),
    "scope": " ".join(REGISTRATION_SCOPES),
}
try:
    response = okta_request("POST", token_endpoint,
        headers={"Content-Type": "application/x-www-form-urlencoded", "Accept": "application/json"},
        data=token_data, allowed=(200,))
    token_body = response_json(response)
    if not isinstance(token_body.get("access_token"), str) or not token_body["access_token"] or str(token_body.get("token_type", "")).lower() != "bearer":
        raise LabError("Token endpoint did not return a usable bearer token.")
    HEADERS = {"Authorization": "Bearer " + token_body["access_token"], "Accept": "application/json"}
    REG_AUTH_ORIGIN = OKTA_DOMAIN
    if OPA_AUTH_MODE == "registration_service" or OPA_AUTH_ORIGIN != OKTA_DOMAIN:
        OPA_HEADERS = dict(HEADERS)
        OPA_AUTH_ORIGIN = OKTA_DOMAIN
        OPA_AUTH_CLIENT_ID = REG_SERVICE_CLIENT_ID
        OPA_USER_AUTHENTICATED = False
        OPA_USER_EXPIRES_AT = 0
        OPA_PENDING = None
    print({"registration_oauth": "PASS", "token_type": token_body.get("token_type"), "expires_in": token_body.get("expires_in")})
finally:
    token_data.clear()
    if "token_body" in globals():
        token_body.clear()


## **Step 3: Verify the existing machine App ID**

Read the existing **machine caller** application and verify its
configuration. This application receives permission to call the agent
through a delegation link in Step 9C. Copy its Okta application ID into
MACHINE_APP_ID; read its OAuth client ID from the response below.

Expected: name, application ID, OAuth client ID and auth method.
Check this app in Admin Console → Applications; do not create or alter it.

[Machine caller prerequisites](https://developer.okta.com/docs/guides/implement-grant-type/clientcreds/main/#set-up-your-app).

---


In [ ]:
guard_config()
invalidate_preflight("app")
if not re.fullmatch(r"[A-Za-z0-9_-]+", MACHINE_APP_ID):
    raise LabError("Set the existing machine application's Okta resource ID.")
response = okta_request("GET", f"/api/v1/apps/{MACHINE_APP_ID}")
SERVICE_APP = response_json(response)
app_settings = SERVICE_APP.get("settings", {}).get("oauthClient", {})
app_credentials = SERVICE_APP.get("credentials", {}).get("oauthClient", {})
if SERVICE_APP.get("id") != MACHINE_APP_ID or SERVICE_APP.get("status") != "ACTIVE" or SERVICE_APP.get("signOnMode") != "OPENID_CONNECT" or app_settings.get("application_type") != "service" or "client_credentials" not in app_settings.get("grant_types", []):
    raise LabError("Choose an active OIDC service app with client_credentials enabled.")
MACHINE_CLIENT_ID = app_credentials.get("client_id")
if not isinstance(MACHINE_CLIENT_ID, str) or not MACHINE_CLIENT_ID:
    raise LabError("The app did not return its OAuth client ID; do not assume an ID mapping.")
# Application responses may contain client_secret; keep only metadata needed later.
app_credentials.pop("client_secret", None)
mark_preflight("app")
print({"app_id": MACHINE_APP_ID, "oauth_client_id": MACHINE_CLIENT_ID,
    "label": SERVICE_APP.get("label"), "auth_method": app_credentials.get("token_endpoint_auth_method")})


---
## **Step 3A: Check the shared server, owner group and new audience**

This cell reads existing resources before creation. It checks the custom
authorization server, the requested runtime scopes, policies that include
the machine caller, and an owner group with at least two members. Policy
metadata is a prerequisite check; the runtime token request later proves
the effective policy decision.

The agent's HTTPS resource URL is its own audience. Do not edit the shared
server's global audience. This lab requires a unique URL for each agent,
although current Okta A2A APIs also allow free-form audience strings.
Existing matching names or resource URLs stop a new registration.

**Expected:** safe server/group identifiers and collision-check success.
**Console:** inspect the existing authorization server and owner group;
this cell makes no changes.

[Custom-server settings, scopes and policies](https://developer.okta.com/docs/guides/customize-authz-server/main/)
· [Understand machine access](https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-add-manually.htm).


---


In [ ]:
invalidate_preflight("machine")
guard_config()
require_preflight("app")
AUTHZ_SERVER = response_json(okta_request(
    "GET", f"/api/v1/authorizationServers/{AUTHZ_SERVER_ID}"
))
if AUTHZ_SERVER.get("status") != "ACTIVE":
    raise LabError("The selected custom authorization server must be ACTIVE.")
if not AUTHZ_SERVER.get("id") or not AUTHZ_SERVER.get("issuer"):
    raise LabError("The authorization server did not return its ID and issuer.")
from urllib.parse import urlsplit

def authorization_issuer() -> str:
    value = AUTHZ_SERVER.get("issuer")
    if not isinstance(value, str):
        raise LabError("The selected AS did not return a valid HTTPS issuer.")
    parsed = urlsplit(value)
    try:
        valid_port = parsed.port in (None, 443)
    except ValueError:
        valid_port = False
    if (
        parsed.scheme != "https" or not parsed.hostname or not valid_port
        or parsed.username is not None or parsed.password is not None
        or parsed.query or parsed.fragment or not parsed.path.startswith("/oauth2/")
        or any(piece in {".", ".."} for piece in parsed.path.split("/"))
    ):
        raise LabError("The authenticated AS metadata must identify an HTTPS Okta custom-AS issuer.")
    return value.rstrip("/")

def oauth_request(method: str, url: str, *, headers: dict[str, str] | None = None, data: dict[str, str] | None = None) -> requests.Response:
    # Issuer metadata came from an authenticated management read.
    # Custom issuer domains are permitted; management bearer credentials
    # must never be forwarded by this separate OAuth transport.
    issuer = authorization_issuer()
    allowed_urls = {
        "POST": {issuer + "/v1/token"},
        "GET": {issuer + "/.well-known/oauth-authorization-server", issuer + "/v1/keys"},
    }
    if url not in allowed_urls.get(method, set()):
        raise LabError("An OAuth endpoint did not match the authenticated AS issuer and expected path.")
    outgoing = dict(headers or {"Accept": "application/json"})
    if any(name.lower() == "authorization" and not value.startswith("Basic ") for name, value in outgoing.items()):
        raise LabError("Management or user bearer credentials cannot be sent by the caller OAuth transport.")
    try:
        with requests.Session() as transport:
            transport.trust_env = False
            response = transport.request(
                method, url, headers=outgoing, data=data,
                timeout=30, allow_redirects=False,
            )
    except requests.RequestException:
        raise LabError("Caller OAuth transport failed; credentials and response details suppressed.") from None
    if response.status_code != 200:
        raise LabError(f"Caller OAuth/metadata request failed (HTTP {response.status_code}); check the caller credentials, scope and AS policy.")
    return response

authorization_issuer()
requested_runtime_scopes = set(RUNTIME_SCOPES.split())
scope_rows = list_all(
    f"/api/v1/authorizationServers/{AUTHZ_SERVER['id']}/scopes"
)
defined_scopes = {row.get("name") for row in scope_rows}
if not requested_runtime_scopes <= defined_scopes:
    raise LabError("A requested runtime scope is not defined on the selected server.")
policy_rows = list_all(
    f"/api/v1/authorizationServers/{AUTHZ_SERVER['id']}/policies"
)
caller_policies = [
    policy for policy in policy_rows
    if policy.get("status") == "ACTIVE"
    and (
        "ALL_CLIENTS" in policy.get("conditions", {}).get("clients", {}).get("include", [])
        or MACHINE_CLIENT_ID in policy.get("conditions", {}).get("clients", {}).get("include", [])
    )
]
if not caller_policies:
    raise LabError("No active authorization-server policy includes the selected machine caller.")
client_credentials_rules = []
for policy in caller_policies:
    rules = list_all(
        f"/api/v1/authorizationServers/{AUTHZ_SERVER['id']}/policies/{policy['id']}/rules"
    )
    client_credentials_rules.extend([
        rule for rule in rules
        if rule.get("status") == "ACTIVE"
        and "client_credentials" in rule.get("conditions", {}).get("grantTypes", {}).get("include", [])
    ])
if not client_credentials_rules:
    raise LabError("No active policy rule permits client_credentials for the selected caller.")

if OWNER_GROUP_ID:
    owner_group = response_json(okta_request(
        "GET", f"/api/v1/groups/{OWNER_GROUP_ID}"
    ))
    if OWNER_GROUP_NAME and owner_group.get("profile", {}).get("name") != OWNER_GROUP_NAME:
        raise LabError("The owner group ID and exact name refer to different groups.")
else:
    owner_matches = [
        group for group in list_all("/api/v1/groups", params={"q": OWNER_GROUP_NAME})
        if group.get("profile", {}).get("name") == OWNER_GROUP_NAME
    ]
    if len(owner_matches) != 1:
        raise LabError("The owner group name must resolve to exactly one group; use its ID if ambiguous.")
    owner_group = owner_matches[0]
RESOLVED_OWNER_GROUP_ID = owner_group["id"]
group_members = list_all(f"/api/v1/groups/{RESOLVED_OWNER_GROUP_ID}/users")
if len({member.get("id") for member in group_members if member.get("id")}) < 2:
    raise LabError("The selected owner group must contain at least two members.")

from urllib.parse import urlsplit
proposed_url = urlsplit(AGENT_RESOURCE_URL)
if (
    proposed_url.scheme != "https" or not proposed_url.hostname
    or proposed_url.username is not None or proposed_url.password is not None
    or proposed_url.fragment or proposed_url.query
):
    raise LabError("Choose a stable HTTPS agent resource URL without credentials, query or fragment.")
known_agent_id = globals().get("AGENT_ID")
matching_names = [
    agent for agent in list_all("/workload-principals/api/v1/ai-agents")
    if agent.get("profile", {}).get("name") == AGENT_NAME
]
# Enumerate all pages: do not assume this endpoint filters an unsupported query.
matching_urls = [
    resource for resource in list_all("/resource-servers/api/v1/a2a-servers")
    if resource.get("resourceUrl") == AGENT_RESOURCE_URL
]
if any(not known_agent_id or agent.get("id") != known_agent_id for agent in matching_names):
    raise LabError("An agent already uses this name. Choose a new name; do not adopt it automatically.")
if any(not known_agent_id or resource.get("a2aServerId") != known_agent_id for resource in matching_urls):
    raise LabError("This resource URL is already registered. Choose a new unique URL.")
mark_preflight("machine")
print(json.dumps({
    "prerequisites": "PASS",
    "authorization_server_id": AUTHZ_SERVER["id"],
    "issuer": AUTHZ_SERVER["issuer"],
    "owner_group_id": RESOLVED_OWNER_GROUP_ID,
    "caller_policy_count": len(caller_policies),
    "client_credentials_rule_count": len(client_credentials_rules),
    "agent_resource_url": AGENT_RESOURCE_URL,
    "collision_check": "PASS",
}, indent=2))


---

## **Step 3B: Discover the existing OPA secret**

This GET lists **metadata**, not secret values. Use the exact secret
ORN in Configuration; preserve its case and partition. We require one
selected existing secret before creating an agent. The registration
service's permissions and an administrator's OPA permissions can differ.

If the service cannot see the selected secret, set `OPA_AUTH_MODE` to
`user_device`, enter the existing native app's client ID, and run Steps
3C–3D. Then rerun this cell. An empty result does not prove that the
tenant contains no secrets. No OPA service key or secret value is needed.

[Potential Connections API](https://developer.okta.com/docs/api/secures-ai/openapi/secures-ai-workload-principals/tags/agentpotentialconnections)


---


In [ ]:
guard_config()
invalidate_preflight("opa")
OPA_PREFLIGHT_OK = False
OPA_SELECTION_CONTEXT = None
if OPA_AUTH_MODE not in {"registration_service", "user_device"}:
    raise LabError("Choose registration_service or user_device for OPA_AUTH_MODE.")

if OPA_AUTH_MODE == "registration_service":
    OPA_HEADERS = dict(HEADERS)
    OPA_AUTH_ORIGIN = OKTA_DOMAIN
    OPA_AUTH_CLIENT_ID = REG_SERVICE_CLIENT_ID
    OPA_USER_AUTHENTICATED = False
    OPA_USER_EXPIRES_AT = 0
    OPA_PENDING = None
else:
    if (
        globals().get("OPA_AUTH_ORIGIN") != OKTA_DOMAIN
        or globals().get("OPA_AUTH_CLIENT_ID") != OPA_DEVICE_CLIENT_ID
        or globals().get("OPA_USER_EXPIRES_AT", 0) <= time.time()
        or not globals().get("OPA_USER_AUTHENTICATED", False)
    ):
        raise LabError("Run Steps 3C and 3D for fresh OPA administrator authorization, then rerun Step 3B.")

_potential_path = "/workload-principals/api/v1/potential-connections"
_potential_rows = list_all(
    _potential_path,
    headers=OPA_HEADERS,
    params={"filter": 'connectionType eq "STS_VAULT_SECRET"'},
)
OPA_SECRET_CHOICES = [
    {
        "name": row.get("secret", {}).get("name", ""),
        "path": row.get("secret", {}).get("path", ""),
        "orn": row.get("secret", {}).get("orn", ""),
    }
    for row in _potential_rows
    if row.get("connectionType") == "STS_VAULT_SECRET"
    and isinstance(row.get("secret"), dict)
    and row["secret"].get("orn")
]
print(json.dumps({"eligible_secret_metadata": OPA_SECRET_CHOICES}, indent=2))
if not OPA_SECRET_CHOICES:
    raise LabError("This identity sees no eligible OPA secret. Check existing-secret access or use Steps 3C–3D; no agent has been created by this prerequisite cell.")
if not OPA_SECRET_ORN or not any(
    row["orn"] == OPA_SECRET_ORN for row in OPA_SECRET_CHOICES
):
    raise LabError("Set OPA_SECRET_ORN to one exact returned ORN, rerun Configuration and prerequisites, and do not create an agent yet.")
OPA_PREFLIGHT_OK = True
OPA_SELECTION_CONTEXT = config_fingerprint()
mark_preflight("opa")
print("OPA prerequisite PASS: the selected existing secret is visible; no secret value was read.")


---

## **Step 3C: Optional administrator sign-in for OPA**

Skip this cell when `OPA_AUTH_MODE` is `registration_service`. Otherwise,
the existing public **native** OIDC app must have Device Authorization
enabled, both AI-agent scopes granted, and your administrator assigned.
Current Beta discovery APIs list **Super Admin** as the required
Okta role; your user also needs the **OPA security admin** role.
The notebook does not create that app or change its permissions.

Run the cell, open the displayed verification URL, enter the user code,
and approve access in Okta. Then run Step 3D. Only OPA discovery and
connection use this user token; registration retains its service token.

**If the browser shows “Device activated,” your notebook sign-in has
been approved.** Return here and run Step 3D to receive the user token.
Activation of the registered AI agent happens later, in Step 10.

Rerunning this cell explicitly starts a new sign-in. Token renewal uses
the same selected agent/settings; it does not create or replace resources.

[Okta Device Authorization](https://developer.okta.com/docs/guides/device-authorization-grant/main/)
· [Assign the intended administrator](https://help.okta.com/oie/en-us/content/topics/provisioning/lcm/lcm-assign-app-user.htm)
· [Discovery API permissions](https://developer.okta.com/docs/api/secures-ai/openapi/secures-ai-workload-principals/tags/agentpotentialconnections)


---


In [ ]:
import re
from urllib.parse import urlsplit

guard_config()
if OPA_AUTH_MODE == "registration_service":
    print("OPA administrator sign-in NOT_RUN: using registration-service authorization.")
elif OPA_AUTH_MODE != "user_device":
    raise LabError("Choose registration_service or user_device for OPA_AUTH_MODE.")
else:
    invalidate_preflight("opa")
    OPA_PREFLIGHT_OK = False
    OPA_SELECTION_CONTEXT = None
    OPA_USER_AUTHENTICATED = False
    OPA_USER_EXPIRES_AT = 0
    OPA_PENDING = None
    OPA_HEADERS = {}
    OPA_AUTH_ORIGIN = ""
    OPA_AUTH_CLIENT_ID = ""
    if not re.fullmatch(r"0oa[A-Za-z0-9]+", OPA_DEVICE_CLIENT_ID):
        raise LabError("Enter the existing native device-flow application's client ID.")
    _device_started = time.monotonic()
    _device_response = None
    _device_json = None
    try:
        _device_response = okta_request(
            "POST", "/oauth2/v1/device/authorize",
            headers={"Accept": "application/json", "Content-Type": "application/x-www-form-urlencoded"},
            data={
                "client_id": OPA_DEVICE_CLIENT_ID,
                "scope": "okta.aiAgents.read okta.aiAgents.manage",
            },
            allowed=(200,),
        )
        _device_json = response_json(_device_response)
        _device_code = _device_json.get("device_code")
        _user_code = _device_json.get("user_code")
        _verification_uri = _device_json.get("verification_uri")
        _expires_in = _device_json.get("expires_in")
        _interval = _device_json.get("interval", 5)
        if (
            not isinstance(_device_code, str) or not _device_code
            or not all(33 <= ord(char) <= 126 for char in _device_code)
            or not isinstance(_user_code, str)
            or not re.fullmatch(r"[A-Za-z0-9-]{1,64}", _user_code)
            or not isinstance(_verification_uri, str)
            or type(_expires_in) is not int or _expires_in <= 0
            or type(_interval) is not int or _interval <= 0
        ):
            raise LabError("Device authorization returned invalid required fields; no token is accepted.")
        _verification_parts = urlsplit(_verification_uri)
        if (
            trusted_url(_verification_uri) != _verification_uri
            or _verification_parts.query or _verification_parts.fragment
            or _device_code in _verification_uri or _device_code in _user_code
        ):
            raise LabError("Device authorization returned an unsafe verification URL or code.")
        _device_deadline = _device_started + _expires_in
        if time.monotonic() >= _device_deadline:
            raise LabError("Device authorization expired before it was received; rerun Step 3C.")
        OPA_PENDING = {
            "origin": OKTA_DOMAIN,
            "client_id": OPA_DEVICE_CLIENT_ID,
            "device_code": _device_code,
            "deadline": _device_deadline,
            "interval": _interval,
            "next_poll": time.monotonic() + _interval,
        }
        print(json.dumps({
            "verification_uri": _verification_uri,
            "user_code": _user_code,
            "expires_in_seconds": _expires_in,
            "poll_interval_seconds": _interval,
            "next_step": "Approve in your browser, then run Step 3D.",
        }, indent=2))
    finally:
        _device_response = _device_json = None
        _device_code = None


---

## **Step 3D: Complete OPA sign-in, then rediscover**

After approving in the browser, run this cell. Its visible loop polls
for at most 40 seconds and honors the server interval, `slow_down`,
expiry, and rate-limit delays. Pending authorization is normal; rerun
this cell to continue the same unexpired grant. Denial or expiry requires
a new explicit Step 3C sign-in. No background task continues afterward.

Success prints token metadata only, then repeats the metadata GET to
validate your selected existing secret. If the upfront ORN is missing
or wrong, copy one displayed ORN into Configuration and rerun the
prerequisites before creating an agent.

[Device polling rules](https://www.rfc-editor.org/rfc/rfc8628.html#section-3.5)


---


In [ ]:
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime

guard_config()
if OPA_AUTH_MODE == "registration_service":
    print("OPA user token NOT_RUN: using registration-service authorization.")
elif OPA_AUTH_MODE != "user_device":
    raise LabError("Choose registration_service or user_device for OPA_AUTH_MODE.")
elif (
    not isinstance(globals().get("OPA_PENDING"), dict)
    or OPA_PENDING.get("origin") != OKTA_DOMAIN
    or OPA_PENDING.get("client_id") != OPA_DEVICE_CLIENT_ID
    or not OPA_PENDING.get("device_code")
):
    raise LabError("Run Step 3C for this org/client, approve in your browser, then run Step 3D.")
else:
    _poll_window_end = min(time.monotonic() + 40, OPA_PENDING["deadline"])
    _user_token_received = False
    _poll_response = _poll_json = None
    try:
        while time.monotonic() < _poll_window_end:
            _poll_wait = max(0, OPA_PENDING["next_poll"] - time.monotonic())
            if _poll_wait:
                time.sleep(min(_poll_wait, max(0, _poll_window_end - time.monotonic())))
            if time.monotonic() >= _poll_window_end:
                break
            OPA_PENDING["next_poll"] = time.monotonic() + OPA_PENDING["interval"]
            _poll_started_epoch = time.time()
            try:
                _poll_response = okta_request(
                    "POST", "/oauth2/v1/token",
                    headers={"Accept": "application/json", "Content-Type": "application/x-www-form-urlencoded"},
                    data={
                        "grant_type": "urn:ietf:params:oauth:grant-type:device_code",
                        "device_code": OPA_PENDING["device_code"],
                        "client_id": OPA_DEVICE_CLIENT_ID,
                    },
                    allowed=(200, 400, 401, 403, 429),
                )
            except LabError:
                OPA_PENDING["interval"] *= 2
                OPA_PENDING["next_poll"] = time.monotonic() + OPA_PENDING["interval"]
                raise
            _poll_json = response_json(_poll_response)
            if _poll_response.status_code == 200:
                _granted = _poll_json.get("scope", "")
                _granted_scopes = _granted.split() if isinstance(_granted, str) else []
                _opa_access_token = _poll_json.get("access_token")
                _opa_expires_in = _poll_json.get("expires_in")
                if (
                    set(_granted_scopes) != {"okta.aiAgents.read", "okta.aiAgents.manage"}
                    or len(_granted_scopes) != 2
                    or not isinstance(_opa_access_token, str) or not _opa_access_token
                    or not all(33 <= ord(char) <= 126 for char in _opa_access_token)
                    or str(_poll_json.get("token_type", "")).lower() != "bearer"
                    or type(_opa_expires_in) is not int or _opa_expires_in <= 0
                    or any(_poll_json.get(name) for name in ("refresh_token", "id_token", "client_secret"))
                ):
                    OPA_PENDING = None
                    raise LabError("OPA token response failed scope/type/expiry validation; restart Step 3C.")
                OPA_HEADERS = {"Authorization": "Bearer " + _opa_access_token, "Accept": "application/json"}
                OPA_AUTH_ORIGIN = OKTA_DOMAIN
                OPA_AUTH_CLIENT_ID = OPA_DEVICE_CLIENT_ID
                OPA_USER_EXPIRES_AT = _poll_started_epoch + _opa_expires_in
                if OPA_USER_EXPIRES_AT <= time.time():
                    OPA_HEADERS = {}
                    OPA_PENDING = None
                    raise LabError("OPA token expired before receipt; restart Step 3C.")
                OPA_USER_AUTHENTICATED = True
                OPA_PENDING = None
                invalidate_preflight("opa")
                OPA_PREFLIGHT_OK = False
                OPA_SELECTION_CONTEXT = None
                _user_token_received = True
                print(json.dumps({
                    "opa_user_authorization": "PASS",
                    "scopes": sorted(_granted_scopes),
                    "expires_in_seconds": _opa_expires_in,
                    "next_step": "Check the following metadata discovery for the selected secret.",
                }, indent=2))
                break
            _poll_error = _poll_json.get("error")
            if _poll_error in {"authorization_pending", "slow_down"} or _poll_response.status_code == 429:
                if _poll_error == "slow_down" or _poll_response.status_code == 429:
                    OPA_PENDING["interval"] += 5
                _retry_after = _poll_response.headers.get("Retry-After", "")
                _retry_delay = 0.0
                try:
                    _retry_delay = max(0.0, float(_retry_after))
                except (ValueError, TypeError):
                    try:
                        _retry_date = parsedate_to_datetime(_retry_after)
                        if _retry_date.tzinfo is None:
                            _retry_date = _retry_date.replace(tzinfo=timezone.utc)
                        _retry_delay = max(0.0, (_retry_date - datetime.now(timezone.utc)).total_seconds())
                    except (ValueError, TypeError, OverflowError):
                        pass
                OPA_PENDING["next_poll"] = time.monotonic() + max(OPA_PENDING["interval"], _retry_delay)
                continue
            OPA_PENDING = None
            if _poll_error == "access_denied":
                raise LabError("OPA authorization was denied. Rerun Step 3C only when you want to authorize again.")
            if _poll_error == "expired_token":
                raise LabError("OPA authorization expired. Rerun Step 3C for a new user code.")
            raise LabError("OPA authorization failed. Check native-client grants, assignment and user permissions before restarting Step 3C.")
        if not _user_token_received:
            if time.monotonic() >= OPA_PENDING["deadline"]:
                OPA_PENDING = None
                raise LabError("OPA authorization expired. Rerun Step 3C for a new code.")
            raise LabError("OPA authorization is still pending. Finish browser approval, then rerun Step 3D; no background polling remains.")
        _user_potential_rows = list_all(
            "/workload-principals/api/v1/potential-connections",
            headers=OPA_HEADERS,
            params={"filter": 'connectionType eq "STS_VAULT_SECRET"'},
        )
        OPA_SECRET_CHOICES = [
            {
                "name": row.get("secret", {}).get("name", ""),
                "path": row.get("secret", {}).get("path", ""),
                "orn": row.get("secret", {}).get("orn", ""),
            }
            for row in _user_potential_rows
            if row.get("connectionType") == "STS_VAULT_SECRET"
            and isinstance(row.get("secret"), dict)
            and row["secret"].get("orn")
        ]
        print(json.dumps({"eligible_secret_metadata": OPA_SECRET_CHOICES}, indent=2))
        if not OPA_SECRET_ORN or not any(row["orn"] == OPA_SECRET_ORN for row in OPA_SECRET_CHOICES):
            raise LabError("This user cannot see the selected ORN. Check OPA security-admin access, or choose a returned existing ORN in Configuration and rerun prerequisites.")
        OPA_PREFLIGHT_OK = True
        OPA_SELECTION_CONTEXT = config_fingerprint()
        mark_preflight("opa")
        print("OPA prerequisite PASS after user-authorized metadata discovery.")
    finally:
        _poll_response = _poll_json = None
        _opa_access_token = None


## **Step 4: Create Agent**

Create the agent using its name, description and immutable
**resourceUrl**. Okta also creates the associated Agent-to-Agent
(A2A) resource. Machine-caller access is configured separately in
Step 9C.

Expected: accepted operation Location, or a confirmed newly created
agent ID. Keep the Location: it identifies asynchronous work, not the
agent. This cell never blindly repeats creation after an uncertain result.

[Agent registration API](https://developer.okta.com/docs/api/secures-ai/openapi/secures-ai-workload-principals/tags/agentregistration)
· [Understand the agent's Console settings](https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-add-manually.htm).

---


In [ ]:
DESIGN_TIME_PASSED = False
RUNTIME_RESULT = {"status": "NOT_RUN", "reason": "configuration_recheck"}
require_preflight("kms", "app", "machine", "opa")
if not globals().get("OPA_PREFLIGHT_OK") or globals().get("OPA_SELECTION_CONTEXT") != config_fingerprint():
    raise LabError("Select and validate an existing OPA secret in Step 3B before creation.")
if AGENT_ID:
    print("An agent ID is already saved; creation skipped. Continue with readback in Step 5.")
elif CREATION_OPERATION_LOCATION:
    print("A creation operation is already saved; continue Step 5 without another POST.")
elif CREATE_ATTEMPTED:
    raise LabError("Creation outcome is uncertain. Inspect Okta by the saved name/URL before deliberate recovery; automatic retry is disabled.")
else:
    LOCKED_CONFIG_FINGERPRINT = config_fingerprint()
    CREATE_ATTEMPTED = True
    agent_payload = {"profile": {"name": AGENT_NAME, "description": AGENT_DESCRIPTION},
                     "resourceUrl": AGENT_RESOURCE_URL}
    response = okta_request("POST", "/workload-principals/api/v1/ai-agents",
        json=agent_payload, allowed=(201, 202))
    if response.status_code == 202:
        if not response.headers.get("Location"):
            raise LabError("Creation was accepted without Location. Inspect Okta before retrying.")
        CREATION_OPERATION_LOCATION = trusted_url(response.headers["Location"])
        CREATED_BY_THIS_NOTEBOOK = True
        print({"creation": "ACCEPTED", "operation_location": CREATION_OPERATION_LOCATION})
    else:
        created = response_json(response)
        AGENT_ID = created.get("id")
        if not isinstance(AGENT_ID, str) or not re.fullmatch(r"[A-Za-z0-9_-]+", AGENT_ID):
            raise LabError("Created response did not contain a valid agent ID. Inspect before retrying.")
        CREATED_BY_THIS_NOTEBOOK = True
        print({"creation": "CREATED", "agent_id": AGENT_ID})


## **Step 5: Get Agent ID from Operation**

Poll the full creation Location until completion, then separately read
the agent and its resource. Record the actual agent ID and agent OAuth
client ID; neither is the operation ID or the machine caller's ID.

Check Admin Console → AI Agents for the exact name/description/URL.
Continue through the remaining configuration and verification steps
before using the agent.

---


In [ ]:
guard_config()
DESIGN_TIME_PASSED = False
RUNTIME_RESULT = {"status": "NOT_RUN", "reason": "configuration_recheck"}
if not AGENT_ID:
    if not CREATION_OPERATION_LOCATION:
        raise LabError("Run Step 4 first; no creation operation is saved.")
    operation_url = trusted_url(CREATION_OPERATION_LOCATION)
    if not urlsplit(operation_url).path.startswith("/workload-principals/api/v1/operations/"):
        raise LabError("Creation returned an unexpected operation path; inspect before proceeding.")
    deadline = time.monotonic() + 90
    while True:
        # Poll the saved creation operation with a bounded wait.
        response = okta_request("GET", operation_url)
        creation_operation = response_json(response)
        operation_status = creation_operation.get("status")
        if operation_status == "COMPLETED":
            break
        if operation_status not in {"SCHEDULED", "IN_PROGRESS"}:
            raise LabError("Creation operation failed or returned an unsupported status.")
        if time.monotonic() >= deadline:
            raise LabError("Creation is still pending; retain its Location and rerun this read-only step later.")
        try:
            wait_seconds = max(2, int(response.headers.get("Retry-After", "2")))
        except ValueError:
            wait_seconds = 2
        time.sleep(min(wait_seconds, max(0, deadline - time.monotonic())))
    candidate_id = creation_operation.get("resource", {}).get("id")
    if not isinstance(candidate_id, str) or not re.fullmatch(r"[A-Za-z0-9_-]+", candidate_id):
        raise LabError("Completed operation did not identify the created agent.")
    AGENT_ID = candidate_id
agent = response_json(okta_request("GET", f"/workload-principals/api/v1/ai-agents/{AGENT_ID}"))
agent_resource = response_json(okta_request("GET", f"/resource-servers/api/v1/a2a-servers/{AGENT_ID}"))
if agent.get("id") != AGENT_ID or agent.get("profile", {}).get("name") != AGENT_NAME or agent.get("profile", {}).get("description") != AGENT_DESCRIPTION or agent_resource.get("a2aServerId") != AGENT_ID or agent_resource.get("resourceUrl") != AGENT_RESOURCE_URL:
    raise LabError("Created resource does not match the reviewed agent configuration.")
AGENT_CLIENT_ID = agent.get("oauthClient", {}).get("clientId")
if not isinstance(AGENT_CLIENT_ID, str) or not AGENT_CLIENT_ID:
    raise LabError("Agent readback did not provide its OAuth client ID.")
print({"agent_id": AGENT_ID, "agent_oauth_client_id": AGENT_CLIENT_ID,
    "resource_url": agent_resource["resourceUrl"], "profile_readback": "PASS"})


## **Step 6: Add Public Key to Agent**

Register the KMS **public** JWK in the agent's credentials endpoint.
Reruns reuse an exact existing active key; a different existing key is
a conflict requiring deliberate recovery, not automatic rotation.

Expected: active matching public key. Check the agent's client binding
in Okta. KMS signing itself remains NOT_RUN until the optional proof.

[Client registration details](https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-add-manually.htm).

---


In [ ]:
guard_config()
DESIGN_TIME_PASSED = False
RUNTIME_RESULT = {"status": "NOT_RUN", "reason": "configuration_recheck"}
if not AGENT_ID or not globals().get("PUBLIC_JWK"):
    raise LabError("Complete Steps 1 and 5 before adding the public key.")
key_path = f"/workload-principals/api/v1/ai-agents/{AGENT_ID}/credentials/jwks"
saved_keys = list_all(key_path)
if saved_keys and not any(same_public_jwk(key, PUBLIC_JWK) and key.get("status") == "ACTIVE" for key in saved_keys):
    raise LabError("Existing agent credentials do not match this KMS key; do not rotate implicitly.")
if not saved_keys:
    response = okta_request("POST", key_path, json=PUBLIC_JWK, allowed=(200, 201))
saved_keys = list_all(key_path)
if not any(same_public_jwk(key, PUBLIC_JWK) and key.get("status") == "ACTIVE" for key in saved_keys):
    raise LabError("Public-key readback did not match an active KMS public JWK.")
print({"agent_id": AGENT_ID, "public_key_kid": PUBLIC_JWK["kid"], "public_key_readback": "PASS"})


## **Step 7: Get Org ID**

Retrieve the organization ID from Okta's public discovery endpoint.
This ID identifies your tenant in the owner, caller and authorization
server resource names (ORNs).

Expected: org ID and partition. Later readbacks verify the actual
references rather than inferring successful binding from string shape.

---


In [ ]:
guard_config()
DESIGN_TIME_PASSED = False
RUNTIME_RESULT = {"status": "NOT_RUN", "reason": "configuration_recheck"}
response = okta_request("GET", "/.well-known/okta-organization", headers={"Accept": "application/json"})
org_data = response_json(response)
ORG_ID = org_data.get("id")
if not isinstance(ORG_ID, str) or not re.fullmatch(r"[A-Za-z0-9_-]+", ORG_ID):
    raise LabError("Organization discovery did not return a valid org ID.")
PARTITION = "oktapreview" if urlsplit(OKTA_DOMAIN).hostname.endswith(".oktapreview.com") else "okta"
print({"org_id": ORG_ID, "partition": PARTITION})


## **Step 8: Get Owner Group ID**

Resolve the selected owner group using its ID or exact name and
confirm that it has at least two members.
This repeats the early prerequisite check so you can inspect the
group immediately before its ownership assignment.

[Create a group](https://help.okta.com/oie/en-us/content/topics/users-groups-profiles/usgp-groups-create.htm)
· [Assign an agent's group owner](https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-add-manually.htm).

---


In [ ]:
guard_config()
DESIGN_TIME_PASSED = False
RUNTIME_RESULT = {"status": "NOT_RUN", "reason": "configuration_recheck"}
if OWNER_GROUP_ID:
    if not re.fullmatch(r"[A-Za-z0-9_-]+", OWNER_GROUP_ID):
        raise LabError("Owner group ID contains unsupported characters.")
    owner_group = response_json(okta_request("GET", f"/api/v1/groups/{OWNER_GROUP_ID}"))
    if OWNER_GROUP_NAME and owner_group.get("profile", {}).get("name") != OWNER_GROUP_NAME:
        raise LabError("Configured owner group ID and exact name do not match.")
    selected_group_id = owner_group.get("id")
else:
    groups = list_all("/api/v1/groups", params={"q": OWNER_GROUP_NAME})
    matches = [g for g in groups if g.get("profile", {}).get("name") == OWNER_GROUP_NAME]
    if len(matches) != 1:
        raise LabError("Group name must resolve to one exact match. Use its explicit ID if names are ambiguous.")
    selected_group_id = matches[0]["id"]
if selected_group_id != RESOLVED_OWNER_GROUP_ID:
    raise LabError("Owner group changed since preflight; inspect before assignment.")
members = list_all(f"/api/v1/groups/{selected_group_id}/users")
if len(members) < 2:
    raise LabError("The selected owner group needs at least two members.")
print({"owner_group_id": selected_group_id, "member_count": len(members)})


## **Step 9: Assign Owner to Agent**

Preserve the explicit governance request: the agent's ORN is the
resource and the group's ORN is its owner principal. This grants
ownership; it is distinct from machine-client delegation.

Expected: exact group ownership readback. Inspect the agent's owner
in the Admin Console. Existing conflicting ownership stops this cell.

---


In [ ]:
guard_config()
DESIGN_TIME_PASSED = False
RUNTIME_RESULT = {"status": "NOT_RUN", "reason": "configuration_recheck"}
if not AGENT_ID or not globals().get("RESOLVED_OWNER_GROUP_ID"):
    raise LabError("Complete agent readback and owner group lookup before assigning ownership.")
agent_orn = orn("directory", f"workload-principals:ai-agents:{AGENT_ID}")
owner_orn = orn("directory", f"groups:{RESOLVED_OWNER_GROUP_ID}")
owner_filter = {"filter": 'resource.type eq "ai-agents" AND parentResourceOrn eq ' + json.dumps(agent_orn)}
owners = list_all("/governance/api/v1/resource-owners", params=owner_filter)
current_principals = {p.get("orn") for row in owners if row.get("resource", {}).get("orn") == agent_orn for p in row.get("principals", [])}
if current_principals and current_principals != {owner_orn}:
    raise LabError("Existing ownership differs from the selected group; deliberate reconciliation is required.")
if not current_principals:
    owner_payload = {"resourceOrns": [agent_orn], "principalOrns": [owner_orn]}
    response = okta_request("POST", "/governance/api/v1/resource-owners", json=owner_payload, allowed=(200, 201, 204))
owners = list_all("/governance/api/v1/resource-owners", params=owner_filter)
current_principals = {p.get("orn") for row in owners if row.get("resource", {}).get("orn") == agent_orn for p in row.get("principals", [])}
if current_principals != {owner_orn}:
    raise LabError("Owner readback did not match the selected group.")
print({"owner_group_id": RESOLVED_OWNER_GROUP_ID, "owner_readback": "PASS"})


---
## **Step 9A: Associate the existing authorization server**

Registering `resourceUrl` already created this agent's A2A resource.
Here you associate the same preexisting custom authorization server used
by your other agents. The request does not create a server, change its
policies, or change its global audience.

The response is asynchronous. Retain its full operation Location and run
Step 9B. Configure this association while the agent is STAGED or INACTIVE,
before Step 10 activates it. An exact ACTIVE association can be reused on rerun. A different
or additional association is a conflict, not permission to replace it.

**Expected:** association accepted, or the exact existing association found.
**Console:** inspect the agent's Machine access authorization server.
[A2A API](https://developer.okta.com/docs/api/secures-ai/openapi/secures-ai-resource-servers/tags/a2aserverregistration)


---


In [ ]:
MACHINE_AS_VERIFIED = False
MACHINE_CONFIGURED = False
DESIGN_TIME_PASSED = False
RUNTIME_RESULT = {"status": "NOT_RUN", "reason": "configuration_recheck"}
guard_config()
require_preflight("machine")
if not globals().get("AGENT_ID"):
    raise LabError("Complete registration and ownership before Step 9A.")
machine_agent = response_json(okta_request(
    "GET", f"/workload-principals/api/v1/ai-agents/{AGENT_ID}"
))
if machine_agent.get("id") != AGENT_ID or machine_agent.get("status") not in {"STAGED", "INACTIVE", "ACTIVE"}:
    raise LabError("The agent read-back must match this run and have a supported lifecycle state.")
machine_resource = response_json(okta_request(
    "GET", f"/resource-servers/api/v1/a2a-servers/{AGENT_ID}"
))
if machine_resource.get("a2aServerId") != AGENT_ID or machine_resource.get("resourceUrl") != AGENT_RESOURCE_URL:
    raise LabError("The agent's persisted A2A resource does not match the reviewed audience.")
MACHINE_AS_ID = AUTHZ_SERVER["id"]
MACHINE_AS_ORN = orn("idp", f"authorization_servers:{MACHINE_AS_ID}")
MACHINE_AS_PATH = f"/resource-servers/api/v1/a2a-servers/{AGENT_ID}/authorization-servers"
associations = list_all(MACHINE_AS_PATH)
if len(associations) > 1 or any(row.get("orn") != MACHINE_AS_ORN for row in associations):
    raise LabError("The agent already has a different or additional authorization server. No changes made.")
MACHINE_AS_VERIFIED = False
if associations:
    if associations[0].get("status") != "ACTIVE":
        if not globals().get("MACHINE_AS_OPERATION_LOCATION"):
            raise LabError("The existing association is not ACTIVE. Inspect its operation before recovery.")
        print("The known association is still pending; continue to Step 9B.")
    else:
        MACHINE_AS_VERIFIED = True
        print("The exact authorization-server association is already ACTIVE. Continue to Step 9B for read-back.")
elif globals().get("MACHINE_AS_OPERATION_LOCATION"):
    print("A prior association operation is known; continue to Step 9B without another POST.")
elif globals().get("MACHINE_AS_ATTEMPTED", False):
    raise LabError("A prior association response was uncertain. Inspect Okta before retrying; no POST repeated.")
else:
    if machine_agent.get("status") == "ACTIVE":
        raise LabError("Configure machine access before activation. This ACTIVE agent has no existing association; no POST or automatic deactivation was performed.")
    MACHINE_AS_ATTEMPTED = True
    association_payload = {"type": "OKTA", "orn": MACHINE_AS_ORN}
    association_response = okta_request(
        "POST", MACHINE_AS_PATH, json=association_payload, allowed=(202,)
    )
    location = association_response.headers.get("Location")
    if not location:
        raise LabError("Association was accepted without an operation Location. Inspect before recovery.")
    candidate_operation_location = trusted_url(location)
    if not urlsplit(candidate_operation_location).path.startswith("/resource-servers/api/v1/operations/"):
        raise LabError("The association returned an unexpected operation URL; do not follow it.")
    MACHINE_AS_OPERATION_LOCATION = candidate_operation_location
    print(json.dumps({"association": "ACCEPTED", "operation_location": MACHINE_AS_OPERATION_LOCATION}, indent=2))


---
## **Step 9B: Resolve the association and read it back**

Follow the returned resource-operation URL with a bounded wait.
Success requires
the selected association itself to exist and be ACTIVE, with the exact
server ID, ORN and issuer. The server list must contain only that server.

If polling reports missing `okta.resourceServers.read`, stop and have the
registration-service grant corrected before renewing its token. A 202,
timeout or failed operation is never a completed machine configuration.

**Expected:** the exact shared AS association passes persisted read-back.


---


In [ ]:
MACHINE_AS_VERIFIED = False
MACHINE_CONFIGURED = False
DESIGN_TIME_PASSED = False
RUNTIME_RESULT = {"status": "NOT_RUN", "reason": "configuration_recheck"}
guard_config()
if not globals().get("MACHINE_AS_PATH"):
    raise LabError("Run Step 9A before resolving its association.")
deadline = time.monotonic() + 90
MACHINE_AS_VERIFIED = False
while time.monotonic() < deadline:
    operation_url = globals().get("MACHINE_AS_OPERATION_LOCATION")
    if operation_url:
        operation_url = trusted_url(operation_url)
        if not urlsplit(operation_url).path.startswith("/resource-servers/api/v1/operations/"):
            raise LabError("The retained association operation URL is invalid; do not follow it.")
        operation_response = okta_request("GET", operation_url)
        operation = response_json(operation_response)
        if operation.get("status") == "FAILED":
            raise LabError("The resource association operation failed; inspect Okta before recovery.")
    association_readback = okta_request(
        "GET", f"{MACHINE_AS_PATH}/{MACHINE_AS_ID}", allowed=(200, 404)
    )
    if association_readback.status_code == 200:
        association = response_json(association_readback)
        if association.get("id") != MACHINE_AS_ID or association.get("orn") != MACHINE_AS_ORN:
            raise LabError("The returned authorization-server association has a different identity.")
        if association.get("issuer") != AUTHZ_SERVER["issuer"]:
            raise LabError("The returned association has an unexpected issuer.")
        if association.get("status") == "ACTIVE":
            MACHINE_AS_VERIFIED = True
            break
        if association.get("status") in {"INVALID", "INACTIVE"}:
            raise LabError("The selected authorization-server association is not operational.")
    if not operation_url:
        raise LabError("No ACTIVE association or known operation exists. Inspect Step 9A before retrying.")
    time.sleep(min(2, max(0, deadline - time.monotonic())))
if not MACHINE_AS_VERIFIED:
    raise LabError("Association polling timed out. Retain its Location and rerun this read-only cell later.")
all_associations = list_all(MACHINE_AS_PATH)
if len(all_associations) != 1 or all_associations[0].get("orn") != MACHINE_AS_ORN or all_associations[0].get("status") != "ACTIVE":
    MACHINE_AS_VERIFIED = False
    raise LabError("Expected exactly one ACTIVE association to the selected authorization server.")
print(json.dumps({"association": "PASS", "authorization_server_id": MACHINE_AS_ID, "issuer": AUTHZ_SERVER["issuer"]}, indent=2))


---
## **Step 9C: Authorize the existing machine caller**

This `ACCESS_TOKEN` delegation authorizes the selected service app to
call this agent. It is separate from linking an OIDC sign-on app for
human user access. The same service app can be a caller for agent two.

`to.authorizationServerOrn` is deliberately omitted from the POST: Okta
derives it from the target's AS association for ACCESS_TOKEN delegation.
The response must identify the server selected in Steps 9A–9B.

**Expected:** exactly one machine-caller link with the correct app,
agent and AS. **Console:** inspect Machine access → callers.
[Delegation Links API](https://developer.okta.com/docs/api/secures-ai/openapi/secures-ai-workload-principals/tags/delegationlinks)


---


In [ ]:
MACHINE_CONFIGURED = False
DESIGN_TIME_PASSED = False
RUNTIME_RESULT = {"status": "NOT_RUN", "reason": "configuration_recheck"}
guard_config()
if not globals().get("MACHINE_AS_VERIFIED", False):
    raise LabError("Complete the exact authorization-server read-back in Step 9B first.")
MACHINE_CONFIGURED = False
caller_orn = orn("idp", f"apps:oidc_client:{MACHINE_APP_ID}")
agent_orn = orn("directory", f"workload-principals:ai-agents:{AGENT_ID}")

def delegation_orn_matches(actual: object, expected: str) -> bool:
    # Observed Preview response alias only; every remaining identity
    # component must still match. Never apply this to OPA secret ORNs.
    return actual == expected or (
        expected.startswith("orn:oktapreview:")
        and actual == "orn:okta:" + expected.removeprefix("orn:oktapreview:")
    )

delegation_path = "/workload-principals/api/v1/delegation-links"
delegation_filter = {"filter": "to.resourceOrn eq " + json.dumps(agent_orn)}
caller_links = list_all(delegation_path, params=delegation_filter)
if len(caller_links) > 1:
    raise LabError("The agent already has multiple caller links; inspect before making any changes.")
if caller_links:
    existing_link = caller_links[0]
    if (
        existing_link.get("from", {}).get("type") != "OKTA_AUTHORIZATION_SERVER"
        or existing_link.get("from", {}).get("tokenType") != "ACCESS_TOKEN"
        or not delegation_orn_matches(existing_link.get("from", {}).get("clientOrn"), caller_orn)
        or not delegation_orn_matches(existing_link.get("to", {}).get("resourceOrn"), agent_orn)
        or not delegation_orn_matches(existing_link.get("to", {}).get("authorizationServerOrn"), MACHINE_AS_ORN)
    ):
        raise LabError("The existing caller link conflicts with the selected machine caller or server.")
    MACHINE_DELEGATION_ID = existing_link["id"]
elif globals().get("MACHINE_DELEGATION_ATTEMPTED", False):
    raise LabError("A prior delegation response was uncertain. Inspect before retrying; no POST repeated.")
else:
    MACHINE_DELEGATION_ATTEMPTED = True
    delegation_payload = {
        "from": {
            "type": "OKTA_AUTHORIZATION_SERVER",
            "clientOrn": caller_orn,
            "tokenType": "ACCESS_TOKEN",
        },
        "to": {"resourceOrn": agent_orn},
    }
    delegation_response = okta_request(
        "POST", delegation_path, json=delegation_payload, allowed=(201,)
    )
    MACHINE_DELEGATION_ID = response_json(delegation_response)["id"]
delegation = response_json(okta_request(
    "GET", f"{delegation_path}/{MACHINE_DELEGATION_ID}"
))
if (
    delegation.get("id") != MACHINE_DELEGATION_ID
    or delegation.get("from", {}).get("type") != "OKTA_AUTHORIZATION_SERVER"
    or delegation.get("from", {}).get("tokenType") != "ACCESS_TOKEN"
    or not delegation_orn_matches(delegation.get("from", {}).get("clientOrn"), caller_orn)
    or not delegation_orn_matches(delegation.get("to", {}).get("resourceOrn"), agent_orn)
    or not delegation_orn_matches(delegation.get("to", {}).get("authorizationServerOrn"), MACHINE_AS_ORN)
):
    raise LabError("The saved delegation did not match the exact caller, agent and server.")
final_links = list_all(delegation_path, params=delegation_filter)
if len(final_links) != 1 or final_links[0].get("id") != MACHINE_DELEGATION_ID:
    raise LabError("Expected exactly one persisted machine caller for this agent.")
final_associations = list_all(MACHINE_AS_PATH)
if len(final_associations) != 1 or final_associations[0].get("orn") != MACHINE_AS_ORN or final_associations[0].get("status") != "ACTIVE":
    raise LabError("The server association changed during caller configuration.")
MACHINE_CONFIGURED = True
print(json.dumps({"machine_access": "PASS", "machine_app_id": MACHINE_APP_ID, "authorization_server_id": MACHINE_AS_ID, "agent_resource_url": AGENT_RESOURCE_URL, "delegation_link_id": MACHINE_DELEGATION_ID}, indent=2))


---

## **Step 9D: Connect the selected existing OPA secret**

The request creates a **reference** to the pre-set secret. It does not
create, copy, reveal, or update its value. The resource indicator defaults
to this exact ORN. Inspect **Directory → AI Agents → your agent → Resource
connections** and verify the same active Secret connection.

A rerun first reads the existing connections: one exact match is reused,
while duplicates or a different secret stop the cell. If user authorization
expires, renew Steps 3C–3D, rerun Step 3B, and retry this cell for the same
agent. A lost POST response is reconciled through the next GET, not blindly
repeated. The registration-service token remains separate.

[Secret connection prerequisites and Console view](https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-connected-resource.htm).


---


In [ ]:
import re

DESIGN_TIME_PASSED = False
RUNTIME_RESULT = {"status": "NOT_RUN", "reason": "configuration_recheck"}
guard_config()
OPA_CONFIGURED = False
if (
    not globals().get("OPA_PREFLIGHT_OK", False)
    or globals().get("OPA_SELECTION_CONTEXT") != config_fingerprint()
):
    raise LabError("Rerun Step 3B for the selected secret before making its connection.")
if OPA_AUTH_MODE == "registration_service":
    OPA_HEADERS = dict(HEADERS)
elif (
    OPA_AUTH_MODE != "user_device"
    or globals().get("OPA_AUTH_ORIGIN") != OKTA_DOMAIN
    or globals().get("OPA_AUTH_CLIENT_ID") != OPA_DEVICE_CLIENT_ID
    or not globals().get("OPA_USER_AUTHENTICATED", False)
    or globals().get("OPA_USER_EXPIRES_AT", 0) <= time.time()
):
    raise LabError("Renew the OPA administrator login in Steps 3C–3D, then rerun Step 3B.")
if not globals().get("AGENT_ID"):
    raise LabError("Complete agent creation and its readback before adding a connection.")
_connections_path = f"/workload-principals/api/v1/ai-agents/{AGENT_ID}/connections"
_connections = list_all(_connections_path, headers=OPA_HEADERS)
_secret_connections = [row for row in _connections if row.get("connectionType") == "STS_VAULT_SECRET"]
if len(_secret_connections) > 1 or any(
    row.get("secret", {}).get("orn") != OPA_SECRET_ORN for row in _secret_connections
):
    raise LabError("The agent has duplicate or conflicting secret connections; inspect it before continuing.")
if _secret_connections:
    _connection_id = _secret_connections[0].get("id")
    print("Reusing the existing exact secret connection for readback.")
else:
    _connection_response = okta_request(
        "POST", _connections_path,
        headers=OPA_HEADERS,
        json={"connectionType": "STS_VAULT_SECRET", "secret": {"orn": OPA_SECRET_ORN}},
        allowed=(200, 201),
    )
    _connection_id = response_json(_connection_response).get("id")
if not isinstance(_connection_id, str) or not re.fullmatch(r"[A-Za-z0-9_-]+", _connection_id):
    raise LabError("The connection response has no safe ID. Inspect the agent before repeating a write.")
OPA_CONNECTION = response_json(okta_request(
    "GET", f"{_connections_path}/{_connection_id}", headers=OPA_HEADERS,
))
if (
    OPA_CONNECTION.get("id") != _connection_id
    or OPA_CONNECTION.get("connectionType") != "STS_VAULT_SECRET"
    or OPA_CONNECTION.get("secret", {}).get("orn") != OPA_SECRET_ORN
    or OPA_CONNECTION.get("resourceIndicator") != OPA_SECRET_ORN
    or OPA_CONNECTION.get("status") != "ACTIVE"
):
    raise LabError("The saved OPA connection does not match the selected active secret reference.")
OPA_CONFIGURED = True
print(json.dumps({
    "opa_connection": "PASS",
    "connection_id": _connection_id,
    "connection_type": "STS_VAULT_SECRET",
    "resource_indicator": OPA_SECRET_ORN,
    "status": "ACTIVE",
    "secret_value_read": False,
}, indent=2))


## **Step 10: Activate Agent**

Activate the agent after completing its machine access and OPA
connection in Steps 9A–9D. Keep the agent STAGED until those steps
have passed, including when working in the Admin Console.

Verify the final ACTIVE state: HTTP 202 means accepted, not completed.
An already active, correctly configured agent is reused on rerun.
The final Step 11 readback must still pass before claiming completion.

---


In [ ]:
guard_config()
DESIGN_TIME_PASSED = False
RUNTIME_RESULT = {"status": "NOT_RUN", "reason": "configuration_recheck"}
if not AGENT_ID:
    raise LabError("Complete agent creation and readback first.")
if not globals().get("MACHINE_CONFIGURED") or not globals().get("OPA_CONFIGURED"):
    raise LabError("Complete Steps 9A–9D machine access and OPA connection while the agent is STAGED before activation.")
agent = response_json(okta_request("GET", f"/workload-principals/api/v1/ai-agents/{AGENT_ID}"))
if agent.get("status") != "ACTIVE":
    response = okta_request("POST", f"/workload-principals/api/v1/ai-agents/{AGENT_ID}/lifecycle/activate", json={}, allowed=(200, 202, 204))
    if response.status_code == 202 and response.headers.get("Location"):
        poll_operation(response.headers["Location"])
deadline = time.monotonic() + 60
while True:
    agent = response_json(okta_request("GET", f"/workload-principals/api/v1/ai-agents/{AGENT_ID}"))
    if agent.get("status") == "ACTIVE":
        break
    if time.monotonic() >= deadline:
        raise LabError("Activation has not read back ACTIVE; inspect before another write.")
    time.sleep(2)
print({"agent_id": AGENT_ID, "status": "ACTIVE", "final_verification": "RUN_STEP_11"})


## **Step 11: Verify the complete onboarding configuration**

Read every persisted requirement again. Earlier success messages are
not substitutes for this check. Exactly one existing machine caller
and the selected existing AS must be bound; this agent keeps its own
immutable URL. Exactly one OPA secret connection must match the chosen
reference and be ACTIVE. Keys and ownership must match the selections.

**PASS here means configuration is complete.** Runtime access and KMS
signing remain independently NOT_RUN until the optional proof succeeds.
Compare these safe results with the agent in the Admin Console.

**Console checkpoint:** open **Directory → AI Agents → your agent**.
Verify its profile, group owner, public-key client registration,
machine-access server/caller and active Secret resource connection.
This machine-access workflow does not configure a human user-access
app; a blank User access section is not a failed requirement here.
[Review agent settings](https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-add-manually.htm)
· [Review resource connections](https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-connected-resource.htm).

---


In [ ]:
guard_config()
DESIGN_TIME_PASSED = False
RUNTIME_RESULT = {"status": "NOT_RUN", "reason": "configuration_recheck"}
if not AGENT_ID or not globals().get("MACHINE_CONFIGURED") or not globals().get("OPA_CONFIGURED"):
    raise LabError("Finish registration, machine access and OPA connection before final verification.")
agent = response_json(okta_request("GET", f"/workload-principals/api/v1/ai-agents/{AGENT_ID}"))
resource = response_json(okta_request("GET", f"/resource-servers/api/v1/a2a-servers/{AGENT_ID}"))
public_keys = list_all(f"/workload-principals/api/v1/ai-agents/{AGENT_ID}/credentials/jwks")
agent_orn = orn("directory", f"workload-principals:ai-agents:{AGENT_ID}")
group_orn = orn("directory", f"groups:{RESOLVED_OWNER_GROUP_ID}")
server_orn = orn("idp", f"authorization_servers:{AUTHZ_SERVER['id']}")
caller_orn = orn("idp", f"apps:oidc_client:{MACHINE_APP_ID}")
ownership = list_all("/governance/api/v1/resource-owners", params={
    "filter": 'resource.type eq "ai-agents" AND parentResourceOrn eq ' + json.dumps(agent_orn)})
principals = {p.get("orn") for row in ownership if row.get("resource", {}).get("orn") == agent_orn for p in row.get("principals", [])}
associations = list_all(f"/resource-servers/api/v1/a2a-servers/{AGENT_ID}/authorization-servers")
delegations = list_all("/workload-principals/api/v1/delegation-links", params={"filter": "to.resourceOrn eq " + json.dumps(agent_orn)})
callers = [link for link in delegations if link.get("from", {}).get("tokenType") == "ACCESS_TOKEN"]
connections = list_all(f"/workload-principals/api/v1/ai-agents/{AGENT_ID}/connections", headers=OPA_HEADERS)
secrets = [row for row in connections if row.get("connectionType") == "STS_VAULT_SECRET"]
DESIGN_TIME_CHECKS = {
    "agent_id": agent.get("id") == AGENT_ID,
    "name": agent.get("profile", {}).get("name") == AGENT_NAME,
    "description": agent.get("profile", {}).get("description") == AGENT_DESCRIPTION,
    "agent_active": agent.get("status") == "ACTIVE",
    "agent_oauth_client_id": agent.get("oauthClient", {}).get("clientId") == AGENT_CLIENT_ID,
    "resource_url_matches": resource.get("a2aServerId") == AGENT_ID and resource.get("resourceUrl") == AGENT_RESOURCE_URL,
    "kms_public_key": any(same_public_jwk(key, PUBLIC_JWK) and key.get("status") == "ACTIVE" for key in public_keys),
    "owner_group": principals == {group_orn},
    "shared_authorization_server": len(associations) == 1 and associations[0].get("orn") == server_orn and associations[0].get("status") == "ACTIVE",
    "one_machine_caller": len(callers) == 1 and callers[0].get("from", {}).get("type") == "OKTA_AUTHORIZATION_SERVER" and delegation_orn_matches(callers[0].get("from", {}).get("clientOrn"), caller_orn) and delegation_orn_matches(callers[0].get("to", {}).get("resourceOrn"), agent_orn) and delegation_orn_matches(callers[0].get("to", {}).get("authorizationServerOrn"), server_orn),
    "one_existing_opa_secret": len(secrets) == 1 and secrets[0].get("secret", {}).get("orn") == OPA_SECRET_ORN and secrets[0].get("resourceIndicator") == OPA_SECRET_ORN and secrets[0].get("status") == "ACTIVE",
}
DESIGN_TIME_PASSED = all(value is True for value in DESIGN_TIME_CHECKS.values())
print(json.dumps({"design_time": "PASS" if DESIGN_TIME_PASSED else "FAIL", "checks": DESIGN_TIME_CHECKS,
    "runtime": RUNTIME_RESULT.get("status", "NOT_RUN")}, indent=2))
if not DESIGN_TIME_PASSED:
    raise LabError("Configuration readback failed. Inspect the named check before continuing.")


## **Optional runtime validation — act as the registered agent**

**Complete Step 11 successfully first.** To validate real access,
set `RUN_RUNTIME_PROOF=True` in Configuration and rerun that cell
with the same resource settings. Add the machine caller's credential
to Colab Secrets, then run **Runtime A followed by Runtime B**. If
validating onboarding only, skip both runtime cells.

Runtime A represents the existing caller. Runtime B represents the
agent receiving that caller's token and requesting its connected
secret. Both roles run in this notebook; no separate agent server
or LLM is required. The private agent key stays in KMS, and the
returned secret values are discarded without being displayed.

**Success:** Runtime A reports `caller_token_verification: PASS`;
Runtime B reports `status: PASS`, `vaulted_secret_received: true`
and `kms_signing_proof: true`. An onboarding PASS alone does not
establish runtime access. A skipped proof remains `NOT_RUN`.

[Read the token-exchange walkthrough](https://developer.okta.com/docs/guides/ai-agent-token-exchange/secret/main/).

---


---
## **RUNTIME A: Obtain the machine caller's token**

The preexisting service app is the caller. It requests a token from the
shared custom AS with `resource` set to **this agent's** URL. This is not
the registration-service management token or the agent's own assertion.

The cell reads the caller's configured authentication method. For
`client_secret_basic` or `client_secret_post`, add Colab Secret
`MACHINE_CLIENT_SECRET`. For `private_key_jwt`, add `MACHINE_PRIVATE_JWK`
matching that caller's registered public key. Its JWT audience is the
custom-AS token endpoint. Never change the existing app's auth method
to make this example run.

[Identify the caller's client authentication method](https://developer.okta.com/docs/api/openapi/okta-oauth/guides/client-auth).

The token is retained in memory and verified with the AS's public JWKS:
signature, issuer, expiry, audience, caller ID and requested scopes.
**Expected:** safe verified claim metadata; no token or secret is printed.
[Token flow](https://developer.okta.com/docs/guides/ai-agent-token-exchange/secret/main/)


---


In [ ]:
CALLER_TOKEN_VERIFIED = False
CALLER_TOKEN = None
CALLER_TOKEN_CONTEXT = None
RUNTIME_RESULT = {"status": "FAIL" if RUN_RUNTIME_PROOF else "NOT_RUN", "passed": False, "caller_token_verified": False, "vaulted_secret_received": False, "secret_value_count": 0, "kms_signing_proof": False}
if RUN_RUNTIME_PROOF:
    guard_config()
    CALLER_TOKEN_VERIFIED = False
    CALLER_TOKEN = None
    if not globals().get("MACHINE_CONFIGURED", False) or globals().get("DESIGN_TIME_PASSED") is not True:
        raise LabError("Complete the machine access configuration and final design-time checks first.")
    current_app = response_json(okta_request("GET", f"/api/v1/apps/{MACHINE_APP_ID}"))
    current_client = current_app.get("credentials", {}).get("oauthClient", {})
    if current_client.get("client_id", current_app.get("id")) != MACHINE_CLIENT_ID:
        raise LabError("The selected machine caller's client identity changed.")
    auth_method = current_client.get("token_endpoint_auth_method", "client_secret_basic")
    discovery = response_json(oauth_request(
        "GET", authorization_issuer() + "/.well-known/oauth-authorization-server",
        headers={"Accept": "application/json"},
    ))
    if discovery.get("issuer") != AUTHZ_SERVER["issuer"]:
        raise LabError("The discovered issuer does not match the selected authorization server.")
    token_endpoint = discovery.get("token_endpoint")
    if token_endpoint != authorization_issuer() + "/v1/token":
        raise LabError("The discovered token endpoint did not match the selected AS issuer.")
    caller_form = {"grant_type": "client_credentials", "resource": AGENT_RESOURCE_URL}
    if RUNTIME_SCOPES.strip():
        caller_form["scope"] = RUNTIME_SCOPES.strip()
    caller_headers = {"Content-Type": "application/x-www-form-urlencoded", "Accept": "application/json"}
    try:
        if auth_method in {"client_secret_basic", "client_secret_post"}:
            caller_secret = get_secret("MACHINE_CLIENT_SECRET")
            if auth_method == "client_secret_basic":
                from urllib.parse import quote
                basic = quote(MACHINE_CLIENT_ID, safe="") + ":" + quote(caller_secret, safe="")
                caller_headers["Authorization"] = "Basic " + base64.b64encode(basic.encode()).decode()
            else:
                caller_form.update(client_id=MACHINE_CLIENT_ID, client_secret=caller_secret)
        elif auth_method == "private_key_jwt":
            try:
                machine_private_jwk = json.loads(get_secret("MACHINE_PRIVATE_JWK"))
            except (ValueError, TypeError):
                raise LabError("MACHINE_PRIVATE_JWK must contain the caller's private JWK as valid JSON.") from None
            caller_form.update(
                client_id=MACHINE_CLIENT_ID,
                client_assertion_type="urn:ietf:params:oauth:client-assertion-type:jwt-bearer",
                client_assertion=client_assertion(MACHINE_CLIENT_ID, token_endpoint, machine_private_jwk),
            )
        else:
            raise LabError("Runtime supports client_secret_basic, client_secret_post or private_key_jwt; the existing app is configured differently.")
        caller_response = oauth_request(
            "POST", token_endpoint, headers=caller_headers, data=caller_form
        )
        caller_body = response_json(caller_response)
        if caller_body.get("token_type", "").lower() != "bearer" or not isinstance(caller_body.get("access_token"), str):
            raise LabError("The caller token endpoint did not return a bearer access token.")
        candidate_token = caller_body["access_token"]
    finally:
        caller_form.clear()
        caller_headers.clear()
        caller_secret = machine_private_jwk = basic = None
    jwks = response_json(oauth_request(
        "GET", discovery["jwks_uri"], headers={"Accept": "application/json"}
    ))
    try:
        token_header = jwt.get_unverified_header(candidate_token)
        matching_keys = [key for key in jwks.get("keys", []) if key.get("kid") == token_header.get("kid")]
        if token_header.get("alg") != "RS256" or len(matching_keys) != 1:
            raise LabError("The caller token's signing key or algorithm could not be verified.")
        caller_claims = jwt.decode(
            candidate_token,
            jwt.PyJWK.from_dict(matching_keys[0], algorithm="RS256").key,
            algorithms=["RS256"], issuer=AUTHZ_SERVER["issuer"], audience=AGENT_RESOURCE_URL,
            options={"require": ["iss", "aud", "exp", "iat"]}, leeway=30,
        )
        if caller_claims.get("aud") not in (AGENT_RESOURCE_URL, [AGENT_RESOURCE_URL]):
            raise LabError("The caller token must target only this agent's resource URL.")
        if caller_claims.get("cid", caller_claims.get("client_id")) != MACHINE_CLIENT_ID:
            raise LabError("The token belongs to a different machine caller.")
        granted_scopes = caller_claims.get("scp", [])
        if not isinstance(granted_scopes, list) or not all(isinstance(scope, str) for scope in granted_scopes):
            raise LabError("The caller token has invalid scope claims.")
        if not set(RUNTIME_SCOPES.split()) <= set(granted_scopes):
            raise LabError("The caller token is missing a requested runtime scope.")
    except jwt.PyJWTError:
        raise LabError("Caller token signature, issuer, audience or expiry verification failed.") from None
    CALLER_TOKEN = candidate_token
    CALLER_TOKEN_CONTEXT = {
        "config_fingerprint": config_fingerprint(),
        "agent_id": AGENT_ID, "client_id": MACHINE_CLIENT_ID,
        "audience": AGENT_RESOURCE_URL, "expires_at": caller_claims["exp"],
    }
    CALLER_TOKEN_VERIFIED = True
    print(json.dumps({
        "caller_token_verification": "PASS", "auth_method": auth_method,
        "issuer": caller_claims["iss"], "audience": AGENT_RESOURCE_URL,
        "caller_client_id": MACHINE_CLIENT_ID, "scopes": granted_scopes,
        "expires_at": caller_claims["exp"],
    }, indent=2))
else:
    print("Caller runtime proof: NOT_RUN. Set RUN_RUNTIME_PROOF only when ready.")


---

## **RUNTIME B: KMS-sign the agent assertion and exchange for its secret**

This optional step validates access as the **registered agent**.
Runtime A supplied the
verified caller token. This cell refreshes AWS credentials, reads the
same KMS public key, verifies its registered binding, and uses `kms:Sign`
without exporting a private key. The assertion audience is the **org
token endpoint**, whereas the caller token audience is the **agent URL**.

The visible token-exchange request uses the exact connection resource
indicator. It retrieves the secret only to verify the response shape;
values are never displayed, saved, or used against a downstream service.
Renew expired AWS credentials in Colab Secrets
and rerun this cell; a changed public key is rejected.

[AWS Sign](https://docs.aws.amazon.com/kms/latest/APIReference/API_Sign.html)
· [Okta secret exchange](https://developer.okta.com/docs/guides/ai-agent-token-exchange/secret/main/)


---


In [ ]:
import uuid

guard_config()
RUNTIME_RESULT = {
    "status": "NOT_RUN", "passed": False,
    "caller_token_verified": False, "vaulted_secret_received": False,
    "secret_value_count": 0, "kms_signing_proof": False,
}
if not RUN_RUNTIME_PROOF:
    print(json.dumps(RUNTIME_RESULT, indent=2))
else:
    RUNTIME_RESULT["status"] = "FAIL"
    if not globals().get("DESIGN_TIME_PASSED", False):
        raise LabError("Complete Step 11's persisted design-time verification before runtime proof.")
    if not globals().get("CALLER_TOKEN_VERIFIED", False) or not globals().get("CALLER_TOKEN"):
        raise LabError("Run Runtime A successfully before requesting the OPA secret.")
    _caller_context = globals().get("CALLER_TOKEN_CONTEXT", {})
    if (
        _caller_context.get("config_fingerprint") != config_fingerprint()
        or _caller_context.get("agent_id") != AGENT_ID
        or _caller_context.get("client_id") != MACHINE_CLIENT_ID
        or _caller_context.get("audience") != AGENT_RESOURCE_URL
        or _caller_context.get("expires_at", 0) <= time.time()
    ):
        raise LabError("The verified caller token is stale or belongs to other settings. Rerun Runtime A.")
    if (
        not globals().get("OPA_CONFIGURED", False)
        or not isinstance(globals().get("OPA_CONNECTION"), dict)
        or OPA_CONNECTION.get("connectionType") != "STS_VAULT_SECRET"
        or OPA_CONNECTION.get("secret", {}).get("orn") != OPA_SECRET_ORN
        or OPA_CONNECTION.get("resourceIndicator") != OPA_SECRET_ORN
        or OPA_CONNECTION.get("status") != "ACTIVE"
    ):
        raise LabError("Runtime proof requires the exact verified active OPA connection from Step 9D.")
    if not isinstance(AGENT_CLIENT_ID, str) or not AGENT_CLIENT_ID:
        raise LabError("Use the agent OAuth client ID returned by agent readback.")
    if not isinstance(KMS_KEY_ARN, str) or not KMS_KEY_ARN.startswith("arn:") or ":key/" not in KMS_KEY_ARN:
        raise LabError("Use the resolved KMS key ARN from Step 1, not an alias.")

    KMS_CLIENT = new_kms_client()
    try:
        _current_key = KMS_CLIENT.get_public_key(KeyId=KMS_KEY_ARN)
    except Exception:
        raise LabError("KMS GetPublicKey failed. Check the key state and renew AWS access for this same key.") from None
    if (
        _current_key.get("KeyId") != KMS_KEY_ARN
        or _current_key.get("KeyUsage") != "SIGN_VERIFY"
        or "RSASSA_PKCS1_V1_5_SHA_256" not in _current_key.get("SigningAlgorithms", [])
    ):
        raise LabError("KMS returned a different key or an incompatible signing configuration.")
    _current_jwk = kms_public_jwk(_current_key["PublicKey"], kid=PUBLIC_JWK["kid"])
    if not same_public_jwk(_current_jwk, PUBLIC_JWK):
        raise LabError("The KMS public key changed. This lab does not rotate the agent's key.")
    _registered_keys = list_all(f"/workload-principals/api/v1/ai-agents/{AGENT_ID}/credentials/jwks", headers=HEADERS)
    if not any(row.get("status") == "ACTIVE" and same_public_jwk(row, PUBLIC_JWK) for row in _registered_keys):
        raise LabError("The selected KMS key is not an active public binding on this agent.")

    _agent_token_endpoint = trusted_url("/oauth2/v1/token")
    _now = int(time.time())
    _assertion_header = {"typ": "JWT", "alg": "RS256", "kid": PUBLIC_JWK["kid"]}
    _assertion_claims = {
        "iss": AGENT_CLIENT_ID, "sub": AGENT_CLIENT_ID,
        "aud": _agent_token_endpoint, "iat": _now,
        "exp": _now + 300, "jti": str(uuid.uuid4()),
    }
    _encoded_header = base64.urlsafe_b64encode(json.dumps(_assertion_header, separators=(",", ":")).encode()).rstrip(b"=").decode()
    _encoded_claims = base64.urlsafe_b64encode(json.dumps(_assertion_claims, separators=(",", ":")).encode()).rstrip(b"=").decode()
    _signing_input = _encoded_header + "." + _encoded_claims
    _digest = hashlib.sha256(_signing_input.encode()).digest()
    _agent_assertion = _exchange_response = _exchange_result = _vaulted_secret = None
    try:
        try:
            _signature_response = KMS_CLIENT.sign(
                KeyId=KMS_KEY_ARN,
                Message=_digest,
                MessageType="DIGEST",
                SigningAlgorithm="RSASSA_PKCS1_V1_5_SHA_256",
            )
        except Exception:
            raise LabError("KMS Sign failed. Check kms:Sign permission, key state and AWS session expiry.") from None
        _signature = base64.urlsafe_b64encode(_signature_response["Signature"]).rstrip(b"=").decode()
        _agent_assertion = _signing_input + "." + _signature
        try:
            _assertion_verified = jwt.decode(
                _agent_assertion,
                jwt.algorithms.RSAAlgorithm.from_jwk(json.dumps(PUBLIC_JWK)),
                algorithms=["RS256"], audience=_agent_token_endpoint,
                issuer=AGENT_CLIENT_ID,
                options={"require": ["iss", "sub", "aud", "iat", "exp", "jti"]},
            )
        except Exception:
            raise LabError("The KMS-signed assertion failed local signature/claims verification.") from None
        if _assertion_verified.get("sub") != AGENT_CLIENT_ID:
            raise LabError("The agent assertion subject does not match its returned client ID.")
        _exchange_response = okta_request(
            "POST", "/oauth2/v1/token",
            headers={"Accept": "application/json", "Content-Type": "application/x-www-form-urlencoded"},
            data={
                "grant_type": "urn:ietf:params:oauth:grant-type:token-exchange",
                "subject_token": CALLER_TOKEN,
                "subject_token_type": "urn:ietf:params:oauth:token-type:access_token",
                "requested_token_type": "urn:okta:params:oauth:token-type:vaulted-secret",
                "resource": OPA_CONNECTION["resourceIndicator"],
                "client_assertion_type": "urn:ietf:params:oauth:client-assertion-type:jwt-bearer",
                "client_assertion": _agent_assertion,
            },
            allowed=(200,),
        )
        _exchange_result = response_json(_exchange_response)
        _vaulted_secret = _exchange_result.get("vaulted_secret")
        if (
            _exchange_result.get("issued_token_type") != "urn:okta:params:oauth:token-type:vaulted-secret"
            or _exchange_result.get("token_type") != "N_A"
            or not isinstance(_vaulted_secret, dict) or not _vaulted_secret
            or not all(isinstance(key, str) and isinstance(value, str) for key, value in _vaulted_secret.items())
        ):
            raise LabError("OPA exchange did not return the expected nonempty vaulted-secret response.")
        RUNTIME_RESULT = {
            "status": "PASS", "passed": True,
            "caller_token_verified": True, "vaulted_secret_received": True,
            "secret_value_count": len(_vaulted_secret), "kms_signing_proof": True,
        }
        print(json.dumps(RUNTIME_RESULT, indent=2))
    finally:
        # Never keep the response object or secret values in notebook globals.
        _exchange_response = _exchange_result = _vaulted_secret = None
        _agent_assertion = None


## **Summary, reruns and recovery**

Record the safe IDs below. Re-run authentication to renew an expired
registration token. Renew OPA user consent in Steps 3C–3D when needed.
For expired AWS sessions, update Colab Secrets: the runtime proof
refreshes AWS access and verifies the same pinned KMS key before signing.

Keep the same public configuration during this run. A resource change
after creation stops later cells. Exact existing keys/owners/callers/
connections can be read and reused; conflicting objects are never
silently replaced. Preserve an uncertain creation Location and inspect
it before considering another POST. After a lost runtime, inspect the
saved agent ID/name/URL; this notebook does not automatically adopt an
existing agent or rotate its keys.

For another new agent, start a fresh runtime or notebook copy, choose
a different name and unique HTTPS URL, and reuse the same existing
machine app and authorization server. Select its intended existing
KMS key, owner and OPA secret. This teaches shared caller/server reuse;
it does not require an Agent A → Agent B → MCP application.

### **What successful completion looks like**

**Onboarding:** Step 11 and this summary report design-time `PASS`,
and the agent's Console settings match your selected resources.
**Runtime access:** after opting in, Runtime A reports caller-token
verification `PASS` and Runtime B reports `PASS` for secret retrieval
and KMS signing. Keep skipped or failed results distinct.

To inspect activity, open **Reports → System Log** or the agent's
**View log** action if available. Select the time of your run and
search for the recorded agent/client identifiers. Review each
relevant event's outcome, actor and target. An empty search alone
does not determine whether an API request succeeded.
[System Log](https://help.okta.com/oie/en-us/content/topics/reports/reports_syslog.htm)
· [System Log filters and search](https://help.okta.com/oie/en-us/content/topics/reports/syslog-filters.htm).

### **If a step stops**

- **Missing Colab Secret:** Exact secret name and Notebook access; place private values in Secrets, not Configuration.
- **Step 1: KMS read:** AWS session expiry, key region/state/type, and DescribeKey/GetPublicKey permissions for the selected key.
- **Step 2: service OAuth:** Registration client ID, matching private/public JWK and `kid`, eight scope grants and app admin role.
- **Step 3A: server, group or audience:** The selected custom server/policy/scopes, two group members, and an unused agent name/resource URL.
- **Step 3B: no visible OPA secret:** Existing secret eligibility and the current identity's access; follow 3C–3D for administrator sign-in when needed.
- **Steps 3C–3D: sign-in pending, expired or denied:** Intended user assignment, Device Authorization, public native client, both scope grants and user/OPA roles. Finish browser approval; start 3C again if expired.
- **Creation or association still pending:** Retain the saved operation Location and rerun its read-only polling step. Check scope/role access; do not repeat creation blindly.
- **Runtime A: caller token:** Existing caller authentication method/credential and effective Client Credentials policy/scopes; use this agent's audience.
- **Runtime B: exchange:** A current caller token, the pinned key's Sign permission, agent public-key binding, delegation and selected ACTIVE Secret connection.

If your administrator needs to investigate, share the **step name,
safe HTTP status, run time and resource/operation IDs**. Keep tokens,
private JWKs, client secrets and secret values out of the report.

---


In [ ]:
guard_config()
print(json.dumps({
    "agent_id": AGENT_ID,
    "agent_oauth_client_id": AGENT_CLIENT_ID,
    "name": AGENT_NAME,
    "agent_resource_url": AGENT_RESOURCE_URL,
    "machine_app_id": MACHINE_APP_ID,
    "machine_oauth_client_id": globals().get("MACHINE_CLIENT_ID"),
    "shared_authorization_server_id": globals().get("AUTHZ_SERVER", {}).get("id"),
    "owner_group_id": globals().get("RESOLVED_OWNER_GROUP_ID"),
    "kms_key_arn": globals().get("KMS_KEY_ARN"),
    "public_key_kid": globals().get("PUBLIC_JWK", {}).get("kid"),
    "opa_connection_id": globals().get("OPA_CONNECTION", {}).get("id"),
    "design_time": "PASS" if DESIGN_TIME_PASSED else "NOT_COMPLETE",
    "runtime": RUNTIME_RESULT.get("status", "NOT_RUN"),
    "kms_signing_proof": "PASS" if RUNTIME_RESULT.get("status") == "PASS" and RUNTIME_RESULT.get("kms_signing_proof") is True else "NOT_PROVEN" if RUNTIME_RESULT.get("status") == "FAIL" else "NOT_RUN",
}, indent=2))


## **Optional appendix: Add a custom authorization-server connection**

An `IDENTITY_ASSERTION_CUSTOM_AS` connection is an **outgoing
resource connection** with its own scope condition. It is different
from the incoming machine-access AS association and from an OPA secret.
This optional connection is separate from the main onboarding
workflow and has its own enable flag and existing scope input.

To explore it separately, ensure your selected custom AS has the
intended existing scope, fill `OPTIONAL_CUSTOM_AS_SCOPE`, and enable
`RUN_OPTIONAL_CUSTOM_AS_CONNECTION`. This does not create or modify
scopes/policies. Leave it disabled unless you need this connection.

---


In [ ]:
guard_config()
if not RUN_OPTIONAL_CUSTOM_AS_CONNECTION:
    print("Custom-AS managed connection: NOT_RUN (optional appendix).")
else:
    if not DESIGN_TIME_PASSED or not OPTIONAL_CUSTOM_AS_SCOPE.strip():
        raise LabError("Finish the main lesson and select an existing appendix scope first.")
    available_scopes = list_all(f"/api/v1/authorizationServers/{AUTHZ_SERVER['id']}/scopes")
    if OPTIONAL_CUSTOM_AS_SCOPE not in {row.get("name") for row in available_scopes}:
        raise LabError("The appendix scope must already exist on the selected custom AS.")
    appendix_payload = {"connectionType": "IDENTITY_ASSERTION_CUSTOM_AS",
        "authorizationServer": {"orn": orn("idp", f"authorization_servers:{AUTHZ_SERVER['id']}")},
        "scopeCondition": "INCLUDE_ONLY", "scopes": [OPTIONAL_CUSTOM_AS_SCOPE]}
    appendix_path = f"/workload-principals/api/v1/ai-agents/{AGENT_ID}/connections"
    existing = [row for row in list_all(appendix_path) if row.get("connectionType") == "IDENTITY_ASSERTION_CUSTOM_AS"]
    if existing and (len(existing) != 1 or any(existing[0].get(k) != v for k, v in appendix_payload.items())):
        raise LabError("Existing custom-AS connection differs; do not replace it implicitly.")
    if not existing:
        okta_request("POST", appendix_path, json=appendix_payload, allowed=(200, 201))
    saved = [row for row in list_all(appendix_path) if row.get("connectionType") == "IDENTITY_ASSERTION_CUSTOM_AS"]
    if len(saved) != 1 or any(saved[0].get(k) != v for k, v in appendix_payload.items()) or saved[0].get("status") != "ACTIVE":
        raise LabError("Optional custom-AS connection readback did not match.")
    print({"optional_managed_connection": "PASS", "connection_id": saved[0]["id"]})


## **Optional cleanup — only the new disposable agent**

Leave `DELETE_CREATED_AGENT=False` if keeping this agent. To remove a
disposable lab agent, inspect the saved ID and current profile, enable
that flag in Configuration, and run this cell. It deactivates/deletes
only the agent confirmed created by this notebook, then requires a
404 readback. It does not delete shared apps, AS policies/scopes, groups,
OPA secrets, sign-in apps or the KMS key.

Before sharing a used copy, clear its outputs and saved form values,
reset input fields and opt-in flags, and end the runtime. Credentials
remain in runtime memory during execution and in your Colab Secrets;
no token or secret value belongs in the shared artifact.

---


In [ ]:
guard_config()
if not DELETE_CREATED_AGENT:
    print("Cleanup: NOT_RUN. Configured agent retained.")
else:
    if not CREATED_BY_THIS_NOTEBOOK or not AGENT_ID:
        raise LabError("Cleanup is limited to the agent confirmed created in this notebook run.")
    agent_path = f"/workload-principals/api/v1/ai-agents/{AGENT_ID}"
    response = okta_request("GET", agent_path, allowed=(200, 404))
    if response.status_code == 200:
        saved = response_json(response)
        if saved.get("id") != AGENT_ID or saved.get("profile", {}).get("name") != AGENT_NAME or saved.get("profile", {}).get("description") != AGENT_DESCRIPTION:
            raise LabError("Cleanup identity check failed; nothing was deleted.")
        if saved.get("status") == "ACTIVE":
            response = okta_request("POST", agent_path + "/lifecycle/deactivate", json={}, allowed=(200, 202, 204))
            if response.status_code == 202 and response.headers.get("Location"):
                poll_operation(response.headers["Location"])
        deadline = time.monotonic() + 60
        while response_json(okta_request("GET", agent_path)).get("status") != "INACTIVE":
            if time.monotonic() >= deadline:
                raise LabError("Deactivation has not completed; deletion stopped.")
            time.sleep(2)
        response = okta_request("DELETE", agent_path, allowed=(200, 202, 204))
        if response.status_code == 202 and response.headers.get("Location"):
            poll_operation(response.headers["Location"])
    deadline = time.monotonic() + 60
    while okta_request("GET", agent_path, allowed=(200, 404)).status_code != 404:
        if time.monotonic() >= deadline:
            raise LabError("Deletion has not read back 404; inspect this agent ID.")
        time.sleep(2)
    DESIGN_TIME_PASSED = False
    RUNTIME_RESULT = {"status": "NOT_RUN", "reason": "agent_deleted"}
    print({"deleted_agent_id": AGENT_ID, "absence_readback": "PASS", "shared_resources": "No delete/update requests issued for them"})


## **Documentation and references**

[Agent registration API](https://developer.okta.com/docs/api/secures-ai/openapi/secures-ai-workload-principals/tags/agentregistration),
[A2A resource API](https://developer.okta.com/docs/api/secures-ai/openapi/secures-ai-resource-servers/tags/a2aserverregistration),
[Delegation links API](https://developer.okta.com/docs/api/secures-ai/openapi/secures-ai-workload-principals/tags/delegationlinks),
[OAuth service apps](https://developer.okta.com/docs/guides/implement-oauth-for-okta-serviceapp/main/),
[Device authorization](https://developer.okta.com/docs/guides/device-authorization-grant/main/),
[OPA secret token exchange](https://developer.okta.com/docs/guides/ai-agent-token-exchange/secret/main/),
[AWS GetPublicKey](https://docs.aws.amazon.com/kms/latest/APIReference/API_GetPublicKey.html),
and [AWS Sign](https://docs.aws.amazon.com/kms/latest/APIReference/API_Sign.html).

Configuration verification and runtime validation report separate outcomes.
Complete all required configuration steps before using the agent; a skipped
or failed check does not establish successful onboarding.
